需要修改的地方：

1. intersector：不计算所有面 只计算相关面

2. Fresnel 输入 n k 计算反射率 （注意是复数情况） 折射率 面吸收率 体吸收率

3. 根据折射路径 计算体吸收概率

4. 完善history： 加入每条光线的反射率 吸收率 折射率 是否全反射 等信息

5. 加入通过筛选接触面找到下个介质的计算方法

6. 测试generate_rays()

7. 尽量完善验证偏振情况

8. n k 反射率 吸收率 折射率等也需要包括入射角度、频率等函数关系





In [ ]:
import numpy as np  # 导入大规模数值计算库，用于全场光线的高效矩阵运算
from scipy.spatial.transform import Rotation as R  # 导入空间三维旋转模块，用于处理复杂的刚体姿态变换

def generate_rays(ray_type="grid", num_rays=100, radius=50, z_start=-300,
                  frequency=5e14, direction_angles=[0, 0], polarization=None,
                  point_source_pos=[0,0,-500], area_size=(100, 100), rotation_euler=[0,0,0],
                  mesh_obj=None, is_index = False):
    """
    生成用于三维光学射线追踪（Ray Tracing）的光线初始参数集，支持多种光源几何拓扑，
    并可选择性地在所有输出矩阵的首列嵌入全局唯一的光线序号（用于多级折反射轨迹追踪）。

    参数:
    ----------
    ray_type : str, 可选
        光源的几何与发射拓扑类型。支持以下五种:
        - "grid": 圆形或方形区域内的均匀网格平行光。
        - "random": 圆形区域内空间随机均匀分布、且方向在球面上各向同性发散的散射光。
        - "point_source": 固定空间点向 4π 全方位均匀辐射的各向同性点光源。
        - "area_source": 具有特定尺寸、支持空间欧拉角旋转的朗伯矩形面光源。
        - "mesh_source": 【全新加入】基于 Trimesh 三角网格的多面体朗伯光源。网格中的每一个三角面片都将
                        作为一个独立的朗伯发射源，其出射光线的半球中心轴严格指向该面的法线正方向。
                        内部采用加权面积无偏采样算法，确保光线通量在不同大小的三角形上物理分布均匀。
        默认为 "grid"。
    num_rays : int, 可选
        期望生成的总光线数量。默认为 100。
    radius : float, 可选
        在 "grid" 或 "random" 模式下，光束横截面的物理覆盖半径。默认为 50。
    z_start : float, 可选
        初始发射面在全局坐标系 Z 轴上的基准位置（对 mesh_source 模式不生效）。默认为 -300。
    frequency : float, 可选
        光波的物理特征参数，在光学追踪中通常代表单色光的波长，单位为纳米 (nm)。默认为 550.0。
    direction_angles : list of float, 可选
        平行光 ("grid") 的标准球坐标系空间传播方向角度对 [theta, phi]，单位为角度 (Degrees)。
        - theta ($\theta$): 天顶角（与全局 $+Z$ 轴的夹角），取值范围 [0, 180]。
        - phi ($\phi$): 方位角（在 $X-Y$ 平面的投影与 $+X$ 轴的夹角），取值范围 [0, 360]。
        默认为 [0, 0]。
    polarization : str 或 int 或 float 或 None, 可选
        光线的初始电场振动状态（偏振态）。支持 "s"、"p" 或表示局部平面夹角的数值(int/float)。
        默认为 None。
    point_source_pos : list of float, 可选
        点光源在全局三维空间中的绝对坐标 [x, y, z]。仅在 `ray_type="point_source"` 时生效。默认为 [0, 0, -500]。
    area_size : tuple of float, 可选
        面光源的二维物理尺寸 (宽度, 高度)。仅在 `ray_type="area_source"` 时生效。默认为 (100, 100)。
    rotation_euler : list of float, 可选
        面光源绕局部坐标轴的外在旋转欧拉角 [X旋, Y旋, Z旋]，单位为角度 (Degrees)。仅在 `ray_type="area_source"` 时生效。默认为 [0, 0, 0]。
    mesh_obj : trimesh.Trimesh, 可选
        【全新加入】输入的 Trimesh 3D 网格对象。仅在 `ray_type="mesh_source"` 时生效。
        函数内部会自动提取其 `mesh_obj.faces`（面索引）、`mesh_obj.vertices`（顶点坐标）以及 `mesh_obj.face_normals`（面法线）。
    is_index : bool, 可选
        是否在返回的各矩阵首列（第 0 列）嵌入 float32 格式的光线唯一索引序号。默认为 False。

    返回:
    -------
    origins, directions, frq, pols : ndarray (float32)
        光线的位置、方向、频率、偏振矩阵。
    """
    # 强制将用户输入的光线总数转换为整型，确保后续底层数组的切片、步长以及分流逻辑合法
    num_rays = int(num_rays)

    # =====================================================================
    # --- 拓扑类型 1: 二维均匀网格平行光束（支持球坐标系双角度倾斜控制） ---
    # =====================================================================
    if ray_type == "grid":
        # 对总光线数开平方并取整，计算出完美的二维正方形网格单边所需的采样点数
        n_side = int(np.sqrt(num_rays))
        # 通过单边采样数的平方重新修正总光线数，确保生成的物理网格绝对对称且没有残缺
        num_rays = n_side ** 2

        # 在设定的光束截面半径范围内 [-radius, +radius]，一维均匀铺开 X 轴的物理坐标采样点
        x = np.linspace(-radius, radius, n_side)
        # 在相同的半径范围内，一维均匀铺开 Y 轴的物理坐标采样点，作为网格的纵列基准
        y = np.linspace(-radius, radius, n_side)

        # 利用 numpy 的 meshgrid 算子，将一维横纵坐标交叉组合，构建出二维空间完整的横纵网格坐标矩阵
        xv, yv = np.meshgrid(x, y)

        # 将二维网格打平为一维序列，并在 Z 轴暂时填入 0，构筑出初始未旋转的局部发射位置矩阵 (N, 3)
        raw_origins = np.stack([xv.flatten(), yv.flatten(), np.zeros(num_rays)], axis=1)

        # 从输入的列表中优雅地解构出球坐标系的方向参数：天顶角 theta_deg 和方位角 phi_deg
        theta_deg, phi_deg = direction_angles

        # 设定在未旋转情况下的基准传播主轴：默认严格沿全局 Z 轴正方向 [0, 0, 1] 垂直向下外扩
        base_dir = np.array([0.0, 0.0, 1.0], dtype=np.float32)

        # 利用欧拉角级联乘法构建旋转算子：先绕局部 Y 轴旋转天顶角 theta，再绕全局 Z 轴旋转方位角 phi
        dir_rot = R.from_euler('z', phi_deg, degrees=True) * R.from_euler('y', theta_deg, degrees=True)
        # 将构建好的高级旋转算子显式展开为标准的 3x3 空间刚体旋转变换矩阵，并强制指定为单精度浮点型
        dir_rot_matrix = dir_rot.as_matrix().astype(np.float32)

        # 运用矩阵乘法（传播矢量乘旋转矩阵的转置），将基准沿 Z 轴的光线方向精确投射到指定的球坐标系空间朝向
        single_direction = base_dir @ dir_rot_matrix.T

        # 利用 tile 算子将这唯一的倾斜传播方向沿行方向平铺复制 N 次，构建出全场完全平行的光束传播方向矩阵
        raw_directions = np.tile(single_direction, (num_rays, 1))

        # 对原始平面网格的每个发射起点也执行完全同步的刚体旋转姿态变换，确保发射面与光束传播方向永远严格垂直
        raw_origins = raw_origins @ dir_rot_matrix.T

        # 在旋转完成的位置矩阵 Z 轴分量上，叠加上全局坐标系下的初始发射面基准物理平移量 z_start
        raw_origins[:, 2] = z_start

    # =====================================================================
    # --- 拓扑类型 2: 圆形发射区域内的空间随机均匀采样、球面发散散射光束 ---
    # =====================================================================
    elif ray_type == "random":
        # 采用逆变换采样法（对随机数开平方再乘半径），从数学上确保在圆形物理截面内实现纯粹的面积均匀分布
        r = radius * np.sqrt(np.random.rand(num_rays))
        # 在 [0, 2π] 弧度区间内为每条散射光线随机、均匀地指派一个极坐标采样方位角
        theta = 2 * np.pi * np.random.rand(num_rays)
        # 结合采样的物理半径与方位角，由极坐标系逆推回直角坐标系，并直接固化所有起点的全局 Z 轴位置为 z_start
        raw_origins = np.stack([r * np.cos(theta), r * np.sin(theta), np.full(num_rays, z_start)], axis=1)

        # 在 [0, 2π] 范围内为发散方向随机采样方位角 phi
        phi = 2 * np.pi * np.random.rand(num_rays)
        # 采样球面天顶角的余弦值，限定在 [-1, 1] 均匀分布，此法是确保高维球面上各向同性（面积均匀）的标准解法
        cos_phi = 2 * np.random.rand(num_rays) - 1
        # 根据三角学第一恒等式，由余弦值严格解算出对应的正弦值，作为空间极径的投影权重
        sin_phi = np.sqrt(1 - cos_phi**2)

        # 运用标准的球坐标系直角化公式，构筑出在全 4π 空间角内各向同性、严格均匀各向发散的单元传播方向矩阵
        raw_directions = np.stack([sin_phi * np.cos(phi), sin_phi * np.sin(phi), cos_phi], axis=1)

    # =====================================================================
    # --- 拓扑类型 3: 位于绝对物理坐标点的各向同性（Isotropic）点光源 ---
    # =====================================================================
    elif ray_type == "point_source":
        # 点光源没有空间发射延展。利用 tile 将用户传入的固定 3D 绝对坐标平铺复制，作为所有光线的共同发射起点
        raw_origins = np.tile(np.array(point_source_pos, dtype=np.float32), (num_rays, 1))

        # 以下发散方向采样的几何物理逻辑与上述 random 拓扑完全一致：先在 [0, 2π] 间均匀随机抽取方位角
        phi = 2 * np.pi * np.random.rand(num_rays)
        # 在 [-1, 1] 区间内均匀抽取天顶角的余弦值，确保点光源向 4π 全球面空间辐射时的通量绝对各向同性
        cos_phi = 2 * np.random.rand(num_rays) - 1
        # 利用平方根三角恒等算式，解算每一条点光源辐射线所对应的高维正弦投影分量
        sin_phi = np.sqrt(1 - cos_phi**2)
        # 将极径投影与方位角进行直角坐标系复合，生成点光源全向辐射的单元传播方向矩阵 (N, 3)
        raw_directions = np.stack([sin_phi * np.cos(phi), sin_phi * np.sin(phi), cos_phi], axis=1)

    # =====================================================================
    # --- 拓扑类型 4: 支持空间任意三维姿态旋转的矩形物理面光源（朗伯余弦发散） ---
    # =====================================================================
    elif ray_type == "area_source":
        # 在局部发射面（ Width 宽度）的对称物理跨度范围 [-W/2, +W/2] 内，对起点进行均匀随机空间采样
        ox = np.random.uniform(-area_size[0]/2, area_size[0]/2, num_rays)
        # 在局部发射面（ Height 高度）的对称物理跨度范围 [-H/2, +H/2] 内，同步对起点的 Y 轴进行均匀采样
        oy = np.random.uniform(-area_size[1]/2, area_size[1]/2, num_rays)
        # 在局部坐标系下，将采样的 X 和 Y 拼接，且令发射面初始暂定在 Z=0 的局部基准面上
        raw_origins = np.stack([ox, oy, np.zeros(num_rays)], axis=1)

        # 在 [0, 2π] 范围内随机抽取面光源出射方向的局部方位角
        phi = 2 * np.pi * np.random.rand(num_rays)
        # 对局部天顶角余弦值进行开方采样（Cos-theta 变换），该数学变换可完美模拟标准的工业级朗伯余弦发散特征
        cos_theta = np.sqrt(np.random.rand(num_rays))
        # 依据三角函数恒等变换公式，精准导出面光源发散方向所需的局部正弦分量权重
        sin_theta = np.sqrt(1 - cos_theta**2)
        # 在面光源的局部上半球空间内 [0, π/2]，组合出符合朗伯辐射定律（中间强、四周弱）的局部各向发射方向
        raw_directions = np.stack([sin_theta * np.cos(phi), sin_theta * np.sin(phi), cos_theta], axis=1)

        # 实例化基于外在固有坐标轴 'xyz' 的顺序欧拉角旋转器，解析用户传入的面光源整体三维姿态调整角
        rot = R.from_euler('xyz', rotation_euler, degrees=True)
        # 将旋转算子展开为 3x3 空间仿射旋转矩阵，并强制类型转换为支持高性能追迹的单精度浮点型
        rot_matrix = rot.as_matrix().astype(np.float32)

        # 运用仿射变换矩阵乘法，将矩形面光源上的所有局部采样起点，整体一次性重投影到全局目标姿态空间中
        raw_origins = raw_origins @ rot_matrix.T
        # 在旋转重映射后的全局起点矩阵 Z 分量上，顺次叠加上物理平移量 z_start，完成最终定位
        raw_origins[:, 2] += z_start
        # 直接通过矩阵乘法对方向矩阵整体旋转，使其姿态与面光源基板完全固联
        raw_directions = raw_directions @ rot_matrix.T

    # =================================================================================
    # --- ⚙️【全新高阶拓扑 5】: 基于 Trimesh 网格的多三角面片自适应并行朗伯光源 ---
    # =================================================================================
    elif ray_type == "mesh_source":
        # 严苛的防御性拦截：如果用户未传入有效的 Trimesh 对象，直接抛出参数缺失错误
        if mesh_obj is None:
            raise ValueError("在 'mesh_source' 模式下，必须指定有效且不为空的 mesh_obj 网格对象！")

        # 1. 物理通量均匀化：提取网格中每一个独立三角面片的物理表面积
        face_areas = mesh_obj.area_faces  # 返回 shape 为 (num_faces,) 的一维面积数组
        # 对全场面积进行累加求和，计算出该实体几何表面网格的总物理辐射面积
        total_area = np.sum(face_areas)
        # 将各个三角面片的面积除以总面积，转化为严格归一化的、可用于蒙特卡洛抽样的概率密度分布 (PDF)
        face_probabilities = face_areas / total_area

        # 依据各个网格三角形的面积概率分布，随机为全场 N 条光线抽取它们各自的归属面索引号
        # 这一步是确保大三角形分配到成比例的更多光线、小三角形分配较少光线的物理核心无偏抽样设计
        chosen_face_indices = np.random.choice(len(face_areas), size=num_rays, p=face_probabilities)

        # 2. 空间几何采样：提取网格中所有顶点的 3D 物理坐标矩阵 (V, 3)
        vertices = np.array(mesh_obj.vertices, dtype=np.float32)
        # 根据抽取的面索引，提取对应的面顶点组合索引矩阵 (N, 3)，其中每一行代表该光线所在的三角形的三点代号
        chosen_faces = mesh_obj.faces[chosen_face_indices]

        # 精准解构映射出全场 N 条光线各自所属三角形的三个物理顶点 A, B, C 坐标矩阵，每个矩阵尺寸均为 (N, 3)
        v0 = vertices[chosen_faces[:, 0]]  # 三角形顶点 A
        v1 = vertices[chosen_faces[:, 1]]  # 三角形顶点 B
        v2 = vertices[chosen_faces[:, 2]]  # 三角形顶点 C

        # 3. 三角形面内无偏面积采样（重心坐标系变换算法）：
        # 首先在 [0, 1] 区间内为所有光线的重心坐标系数随机抽取两个独立的均匀标量数组
        u1 = np.random.rand(num_rays)
        u2 = np.random.rand(num_rays)

        # 🔒【几何拓扑硬约束】：对 u1 执行开方操作。这是重心坐标系实现纯面积均匀、消除边界聚集的经典无偏修正数学公式
        sqrt_u1 = np.sqrt(u1)
        # 解算出三角形三个顶点在组合出空间坐标时各自承担的物理权重系数（满足 alpha + beta + gamma = 1.0 刚性约束）
        alpha = 1.0 - sqrt_u1
        beta = sqrt_u1 * (1.0 - u2)
        gamma = sqrt_u1 * u2

        # 运用重心坐标系线性加权算式，并行解算出 N 条光线在网格实体表面各自对应的 3D 初始发射位置矩阵
        raw_origins = (alpha[:, None] * v0 + beta[:, None] * v1 + gamma[:, None] * v2).astype(np.float32)

        # 4. 朗伯半球出射方向并行构造：
        # 顺次提取全场 N 条光线归属三角面片对应的单位法向量，作为朗伯发射的局部半球中心主轴 (N, 3)
        normals = np.array(mesh_obj.face_normals[chosen_face_indices], dtype=np.float32)

        # 为每条光线的朗伯散射，在其法线所在的局部上半球坐标系内进行标准的 Cos-theta 概率余弦采样
        phi_lambert = 2 * np.pi * np.random.rand(num_rays)      # 局部半球内的方位角
        cos_theta_lambert = np.sqrt(np.random.rand(num_rays))   # 对余弦值开方以完美契合 Lambertian 漫反射物理特征
        sin_theta_lambert = np.sqrt(1.0 - cos_theta_lambert**2) # 解算出对应的局部正弦投影项

        # 组合出局部半球坐标系下（即暂定法线指向 [0, 0, 1] 时）的临时散射方向单元矢量
        local_dirs = np.stack([
            sin_theta_lambert * np.cos(phi_lambert),
            sin_theta_lambert * np.sin(phi_lambert),
            cos_theta_lambert
        ], axis=1).astype(np.float32)

        # 5. 局部朗伯方向向全局面法线朝向的非旋转刚体坐标转换：
        # 为了将 local_dirs 投射到真正的面法线 normals 上，需要为每个三角面快速动态构建正交基（T, B, N）
        # 选择一个绝不与法线重合的辅助参考向量：若法线的 X 绝对值小于 0.5，则选 [1,0,0]，否则选 [0,1,0]
        up_vectors = np.where(np.abs(normals[:, 0:1]) < 0.5,
                              np.array([1.0, 0.0, 0.0], dtype=np.float32),
                              np.array([0.0, 1.0, 0.0], dtype=np.float32))

        # 运用两次向量叉乘，快速解算出与当前三角形面法线严格正交切平面的切向量（Tangent）与副切向量（Bitangent）
        tangents = np.cross(up_vectors, normals)
        tangents /= np.linalg.norm(tangents, axis=1, keepdims=True)  # 切向量规范单位化
        bitangents = np.cross(normals, tangents)                     # 依据右手定则自动导出副切向量

        # 通过正交基基底线性变换矩阵乘法，将局部半球采样方向一次性并行重投影回真正的网格表面法线空间中
        raw_directions = (local_dirs[:, 0:1] * tangents +
                          local_dirs[:, 1:2] * bitangents +
                          local_dirs[:, 2:3] * normals).astype(np.float32)

        # 确保输出的最终传播方向矩阵分量纯净，再次执行行向量单位化提纯
        raw_directions /= np.linalg.norm(raw_directions, axis=1, keepdims=True)

    else:
        # 如果传入的拓扑识别码无法被引擎解析，说明属于非法调用，立即主动抛出数据异常错误终止程序
        raise ValueError(f"未知的光源类型: {ray_type}")

    # =====================================================================
    # --- 数据高阶变阵与序号嵌入块（全自动多级光线轨迹关联追迹机制） ---
    # =====================================================================
    # 创建一个单调递增的、从 0 到 N-1 的一维全场光线唯一索引序列，并将其强制重塑为 (N, 1) 的二维列向量
    ray_indices = np.arange(num_rays, dtype=np.float32).reshape(-1, 1)

    # 依据 is_index 开关状态：若为真，则利用水平拼接将唯一光线序号植入矩阵第 0 列；若为假，则保留原 (N, 3) 数据
    origins = np.hstack([ray_indices, raw_origins]) if is_index else raw_origins
    # 对单元传播方向矩阵执行完全一致的序号挂载拦截逻辑，确保几何坐标与方向分量在同一条光线下严格保序绑定
    directions = np.hstack([ray_indices, raw_directions]) if is_index else raw_directions

    # 在全场平铺分配光线的物理特征列向量。通常填充单色光的中心波长（如 550nm 绿光），转为 (N, 1) 的二维列向量
    raw_frq = np.full((num_rays, 1), float(frequency), dtype=np.float32)
    # 对特征波长/频率矩阵同步实施光线序号嵌入策略，为后续多光谱色散、色差仿真计算提供核心索引保障
    frq = np.hstack([ray_indices, raw_frq]) if is_index else raw_frq

    # =====================================================================
    # --- 核心物理块: 具有几何退化边界保护机制的动态本征/夹角偏振矢量处理 ---
    # =====================================================================
    if polarization is not None:
        if isinstance(polarization, (str, int, float)):
            z_axis = np.array([0, 0, 1], dtype=np.float32)
            degenerate = (np.abs(raw_directions[:, 0]) < 1e-6) & (np.abs(raw_directions[:, 1]) < 1e-6)

            s_dirs = np.cross(raw_directions, z_axis)
            norms = np.linalg.norm(s_dirs, axis=1, keepdims=True)
            norms = np.where(norms == 0, 1.0, norms)
            s_dirs = s_dirs / norms
            s_dirs[degenerate] = [1.0, 0.0, 0.0]

            p_dirs = np.cross(s_dirs, raw_directions)
            p_norms = np.linalg.norm(p_dirs, axis=1, keepdims=True)
            p_dirs = p_dirs / np.where(p_norms == 0, 1.0, p_norms)

            if polarization == "s":
                raw_pols = s_dirs
            elif polarization == "p":
                raw_pols = p_dirs
            else:
                angle_rad = np.radians(float(polarization))
                raw_pols = np.cos(angle_rad) * s_dirs + np.sin(angle_rad) * p_dirs
        else:
            raise ValueError("偏振参数输入不合法！请传入 's'、'p' 或代表局部偏振角的数值(int/float)。")

        pols = np.hstack([ray_indices, raw_pols]).astype(np.float32) if is_index else raw_pols.astype(np.float32)
    else:
        pols = None

    return origins.astype(np.float32), directions.astype(np.float32), frq.astype(np.float32), pols

<>:35: SyntaxWarning: invalid escape sequence '\p'
<>:35: SyntaxWarning: invalid escape sequence '\p'
/tmp/ipykernel_361/468781287.py:35: SyntaxWarning: invalid escape sequence '\p'
  - phi ($\phi$): 方位角（在 $X-Y$ 平面的投影与 $+X$ 轴的夹角），取值范围 [0, 360]。


输入为

origins : ndarray of shape (N, 4)
        光线的起始坐标矩阵。数据类型为 float32。
        每一行的四列映射为: [光线序号, X坐标, Y坐标, Z坐标]
    directions : ndarray of shape (N, 4)
        光线的单元传播方向导数矩阵。矢量已做归一化处理处理，数据类型为 float32。
        每一行的四列映射为: [光线序号, dX分量, dY分量, dZ分量]
    wls : ndarray of shape (N,)
        光线的波长一维数组。数据类型为 float32。
    pols : ndarray of shape (N, 4) 或 None
        光线的初始偏振状态矩阵。若输入参数 `polarization` 为 None，则返回 None。
        每一行的四列映射为: [光线序号, pX分量, pY分量, pZ分量]

与 一些stl物体文件 每个stl物体有单独的名字，还有反射率 r_map、吸收率 a_map

反射率 + 吸收率 = 1

追踪每条光线的反射或吸收 如果吸收 终止该条光线tracing

定义x_lim, y_lim, z_lim, 如果光线超出这些limits，停止计算，将光线截止在limits

输出ray_history记录每次反射光线的起点，每条光线结构为：

[光线序号，起点坐标1，所在介质1名字，起点坐标2，所在介质2名字，。。。]

还有输出ray_status,记录光线现在的状态，每条光线为：

[光线序号, 最后一次起点坐标，最后一次光线方向，最后一次介质名字]

如果最后一次光线防线为(0, 0, 0) 表示光线被吸收

用pytorch在GPU计算

===

请扩展加入折射、漫反射还有fresnel折射反射 还有与频率相关的变化率

折射率、反射率、吸收率为光频率相关函数：
	如果是一个单一数值，那么就是该固定值
	如果是一个函数，就是频率相关函数
	如果是一个dict = {频率： 数值} 那么就用interpolation或者EXTRAPOLATION获得相关数值

	 折射率 + 反射率 + 吸收率 = 1

如果polarization为None 按照普通折射反射计算

如果指定polarization：反射率 = 基本反射率*fresnel反射率

然后根据反射率，和吸收率，计算折射率： 折射率 + 反射率 + 吸收率 = 1

保证运算正确和高性能


In [ ]:
!pip install numpy numpy-stl

In [ ]:
import os
import torch
import numpy as np

# =====================================================================
# 0. 核心计算硬件的自适应绑定与环境配置
# =====================================================================
# 检查当前系统是否有可用的 NVIDIA CUDA GPU，有则使用 'cuda'，否则回退到 'cpu'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# 在终端打印当前光追引擎实际绑定的计算设备
print(f"核心物理计算引擎已绑定至设备: {device}")


# =====================================================================
# 1. 纯张量化并行 1D 线性插值与两端外推算子
# =====================================================================
def gpu_interp1d(x_queries, x_data, y_data, div_epsilon=1e-8):
    """
    在不借助任何 Python 循环的前提下，利用张量并行架构，
    为成千上万条不同频率的光线，从离散的材质采样点中解算出当下的物理特征（如折射率）。
    """
    # 1. 对原始离散采样点的X坐标进行升序排序，返回其对应的索引组合
    sort_idx = torch.argsort(x_data)
    # 2. 根据排序索引，对X采样点进行重排，确保其严格单调递增
    x_d = x_data[sort_idx]
    # 3. 保持映射一致性，对Y采样点（对应的物理响应值）进行相同的重排
    y_d = y_data[sort_idx]

    # 4. 核心算子：利用二分查找将待查询的X高效分桶，确定每条光线落在哪个离散区间内
    idx = torch.bucketize(x_queries, x_d)
    # 5. 防御边界溢出：将索引截断在 [1, N-1] 之间，确保后续 idx-1 和 idx 均合法，同时处理了外推
    idx = torch.clamp(idx, 1, len(x_d) - 1)

    # 6. 提取插值区间左端点的 X 坐标
    x0 = x_d[idx - 1]
    # 7. 提取插值区间右端点的 X 坐标
    x1 = x_d[idx]
    # 8. 提取插值区间左端点的 Y 响应值
    y0 = y_d[idx - 1]
    # 9. 提取插值区间右端点的 Y 响应值
    y1 = y_d[idx]

    # 10. 计算该区间的割线斜率（k = Δy / Δx），并加入极小项防止分母为 0 导致 NaN
    slopes = (y1 - y0) / (x1 - x0 + div_epsilon)
    # 11. 执行标准一阶线性插值公式：y = y0 + k * (x - x0)，解出查询点的物理值
    y_queries = y0 + slopes * (x_queries - x0)
    # 12. 返回并行插值计算出的结果张量
    return y_queries


class SimpleMeshIntersector:
    def __init__(self, vertices, faces, tri_to_obj_ids, div_epsilon=1e-9, front_epsilon=1e-4, adaptive_error_epsilon=1e-6):
        """
        高性能构造函数：直接接收固化在指定显存/内存中的 PyTorch Tensor。
        利用张量高级索引与行内连续性优化，实现零冗余拷贝的几何属性预计算。
        """
        self.div_epsilon = div_epsilon        # 预设的除法防御极小量，规避光线与三角形平行时分母为零
        self.front_epsilon = front_epsilon    # 预设的光线推进时防止自交的硬性平移偏差（近剪裁平面）
        self.adaptive_error_epsilon = adaptive_error_epsilon  # 动态误差计算的基底极小量

        # 🟢 PBRT 严密推导得出的单精度浮点数机器 Epsilon 累积系数 gamma_3
        # 对应 PBRT 中的 gamma(3) = (3 * machine_epsilon) / (1 - 3 * machine_epsilon)
        self.gamma_3 = 5.960464e-7

        # 🟢 核心加速点 1：动态锚定主设备树（Device Anchor）
        # 强制断言输入必须是张量。直接读取顶点所在的硬件（如 'cuda:0' 或 'cpu'），作为全场景的物理主设备
        assert isinstance(vertices, torch.Tensor), "输入 vertices 必须是 PyTorch 张量。"


        # 🟢 核心加速点 2：内存连续性拉直（Contiguous Allocation）
        # .detach() 斩断上游可能存在的反向传播梯度图，释放多余内存
        # .contiguous() 把可能因切片、过滤导致零碎的显存地址拉直为连续排列，这是底层 CUDA 并行高吞吐读取的前提
        self.vertices = vertices.detach().contiguous().to(dtype=torch.float32)

        # 将拓扑面片和上游合并生成的 [接收方, 发射方] 二维标签矩阵同步安全搬运至相同的硬件设备上
        self.faces = faces.detach().contiguous().to(dtype=torch.long, device=device)
        self.tri_to_obj_ids = tri_to_obj_ids.detach().contiguous().to(dtype=torch.long, device=device)

        # 🟢 核心加速点 3：利用高级索引（Advanced Indexing）在显存内部并行抽离所有三角形的三个顶点坐标
        self.tri_v0 = self.vertices[self.faces[:, 0]]  # 提取所有三角形的第一个顶点坐标，形状为 (T, 3)，T 为三角形总数
        self.tri_v1 = self.vertices[self.faces[:, 1]]  # 提取所有三角形的第二个顶点坐标，形状为 (T, 3)
        self.tri_v2 = self.vertices[self.faces[:, 2]]  # 提取所有三角形的第三个顶点坐标，形状为 (T, 3)

        # 全矩阵并行化计算三角形的两条局部空间边界向量
        e1 = self.tri_v1 - self.tri_v0  # 从 v0 指向 v1 的边向量，形状为 (T, 3)
        e2 = self.tri_v2 - self.tri_v0  # 从 v0 指向 v2 的边向量，形状为 (T, 3)

        # 向量叉乘：通过两条边向量求出整个场景所有三角形未归一化的原始几何面法线 (T, 3)
        raw_normals = torch.cross(e1, e2, dim=-1)

        # 对法线进行 L2 范数归一化，使其成为标准的单位方向向量。末尾加上 div_epsilon 防止退化三角形引发分母为零：输出为所有三角形的normals
        self.tri_normals = raw_normals / (torch.norm(raw_normals, dim=-1, keepdim=True) + self.div_epsilon)

        # 记录并固化该场景多边形网格中的三角形总数
        self.num_triangles = self.tri_v0.shape[0]

    def intersect_batch(self, ray_o, ray_d, prev_hit_face_ids):
        """
        全张量并行交点推演：依靠 PyTorch 升维广播，实现 R 条光线与 T 个三角形的同时求交（完全消除 Python 循环）。
        返回：
            d_hit: (R, 3) 击中光线的入射方向向量（保持健康二维方向矩阵）
            normals: (R, 3) 击中点处物体的几何单位法线
            obj_ids: (R, 2) 二维物理/材质看板 [接收方 mesh_id, 发射方 mesh_id]
            hit_points: (R, 3) 击中地点的世界坐标
            p_error: (R, 3) 工业级自适应浮点数机器误差包络线
            hit_mask: (R,) 全局布尔命中掩码（用于外部追踪分流）
            min_tri_idx: (R,) 本次命中的三角形在场景大表中的真实行号（用于状态机下轮交接）
        """
        R = ray_o.shape[0]        # 获取当前批次输入的光线总数 R
        T = self.num_triangles    # 获取场景中的三角形总数 T

        # =========================================================================
        # 1. 关键升维：利用轴扩张（unsqueeze）构造满足 PyTorch 自动广播机制的求交矩阵空间
        # =========================================================================
        o = ray_o.unsqueeze(1)                # 光线起点：由 (R, 3) 扩张为 (R, 1, 3)
        d = ray_d.unsqueeze(1)                # 光线方向：由 (R, 3) 扩张为 (R, 1, 3)
        v0 = self.tri_v0.unsqueeze(0)          # 三角形顶点 v0：由 (T, 3) 扩张为 (1, T, 3)

        # 计算每个三角形的边向量并同样升维至 (1, T, 3)
        e1 = (self.tri_v1 - self.tri_v0).unsqueeze(0)
        e2 = (self.tri_v2 - self.tri_v0).unsqueeze(0)

        # =========================================================================
        # 2. Möller-Trumbore 算法核心步骤的矩阵向量化并行操作
        # =========================================================================
        # 计算光线方向 d 与边向量 e1 的叉乘张量 p。通过 expand 机制强制将 e1 广播拷贝放大为 (R, T, 3)
        p = torch.cross(d, e1.expand(R, T, 3), dim=-1)

        # 计算行列式 det：边向量 e2 与张量 p 的点积，点积后空间维度缩减，整体形状变为 (R, T)
        det = torch.sum(e2 * p, dim=-1)

        # 数值硬性防御：若行列式的绝对值接近 0，说明光线与三角形平面平行。将其强制锁定在微量，避免除以 0 导致溢出
        det = torch.where(torch.abs(det) < self.div_epsilon, torch.sign(det) * self.div_epsilon, det)
        inv_det = 1.0 / det                   # 求出行列式的倒数

        s = o - v0                            # 计算从三角形顶点 v0 指向光线起点的距离向量空间，形状自动广播为 (R, T, 3)
        u = torch.sum(s * p, dim=-1) * inv_det # 求解 Möller-Trumbore 重心坐标 u，形状为 (R, T)

        # 计算向量 s 与边向量 e2 的叉乘张量 q，通过 expand 并行拷贝放大为 (R, T, 3)
        q = torch.cross(s, e2.expand(R, T, 3), dim=-1)
        v = torch.sum(d * q, dim=-1) * inv_det # 求解 Möller-Trumbore 重心坐标 v，形状为 (R, T)
        t = torch.sum(e1 * q, dim=-1) * inv_det # 求解交点参数 t（即光线自由运动直到撞击的物理距离），形状为 (R, T)

        # =========================================================================
        # 3. 向量化遮罩过滤：判定交点是否落在三角形内，并执行自相交免疫
        # =========================================================================
        # 几何判定布尔掩码：重心坐标必须满足 (u >= 0, v >= 0, u+v <= 1)，且推进距离必须大于防自交硬性阈值
        valid_mask = (u >= 0.0) & (v >= 0.0) & ((u + v) <= 1.0) & (t > self.front_epsilon)

        # 🚀 拓扑级自相交物理屏蔽：利用 Tensor 内存行号看板彻底抹去 Python 循环
        # 生成一个覆盖场景所有三角形的本地位置行号索引看板，形状为 (1, T)
        scene_face_indices = torch.arange(T, device=device, dtype=torch.int64).unsqueeze(0)
        prev_ids_2d = prev_hit_face_ids.view(R, 1) # 将传入的上轮命中行号拉直并升维至 (R, 1)

        # 构造自相交屏蔽矩阵：如果当前正在计算的三角形行号正是该光线上轮命中的行号(且排除初始化的-1)，则触发强屏蔽
        self_intersect_mask = (scene_face_indices == prev_ids_2d) & (prev_ids_2d >= 0)

        # 利用对布尔掩码取反（~），将属于自相交面片的有效位置强行熔断重置为 False
        valid_mask = valid_mask & (~self_intersect_mask)

        # 创建一个全为正无穷大（inf）的几何缓冲张量，形状与距离矩阵 t 一致 (R, T)
        inf_t = torch.full_like(t, float('inf'))

        # 距离空间过滤：如果掩码有效（确实撞击且合法）则保留计算的距离 t，否则一律视为正无穷大（没撞到）
        t_to_search = torch.where(valid_mask, t, inf_t)

        # 🟢 硬件加速降维并行归约：在场景所有三角形维度（dim=-1）上寻找数值最小的交点距离
        # min_t 吐出最近撞击处的距离 (R,)；min_tri_idx 吐出的正是本次撞击的三角形在张量中的【真实行号位置】(R,)
        min_t, min_tri_idx = torch.min(t_to_search, dim=-1)

        # 生成最终的全局命中布尔掩码：只要最近距离不是无穷大，说明该光线成功击中了场景中的某个几何面
        hit_mask = min_t != float('inf')

        # =========================================================================
        # 4. 二维物性/材质看板并行提取（完美承接上游 [接收方, 发射方] 映射表）
        # =========================================================================
        # 依据命中三角形的真实行号，利用高级索引一举抽取完整的两列物性数据，形状为 (R, 2)
        hit_obj_categories = self.tri_to_obj_ids[min_tri_idx, :]

        # 利用布尔广播，如果光线未击中任何物体，则该光线被稳固赋予 [-1, -1] 的未命中临时成绩单
        obj_ids = torch.where(hit_mask.unsqueeze(-1), hit_obj_categories, torch.full_like(hit_obj_categories, -1))

        # 预先在显存中开辟全零的法线结果大张量 (R, 3) 与世界坐标交点大张量 (R, 3)
        normals = torch.zeros((R, 3), dtype=torch.float32, device=device)
        hit_points = torch.zeros((R, 3), dtype=torch.float32, device=device)

        # 硬件执行分支优化：只要当前批次中有任意一条光线击中几何体，就进入内部并行提取详细物理量
        if hit_mask.any():
            # 批量提取被击中三角形在初始化时存好的固有单位法向量
            normals[hit_mask] = self.tri_normals[min_tri_idx[hit_mask]]
            # 经典射线线性方程代入：P = O + t * D，一步算出所有命中光线交点的精确世界坐标
            hit_points[hit_mask] = ray_o[hit_mask] + min_t[hit_mask].unsqueeze(-1) * ray_d[hit_mask]

        # =========================================================================
        # 5. 【PBRT 工业级自适应机器浮点数误差包络线计算】
        # =========================================================================
        # 依据交点坐标的绝对值大小，自适应计算单精度浮点数在长距离空间运动中的误差积累包络
        adaptive_error = self.gamma_3 * torch.abs(hit_points) + self.adaptive_error_epsilon
        # 若光线未击中任何物体，则其误差矢量归零
        p_error = torch.where(hit_mask.unsqueeze(-1), adaptive_error, torch.zeros_like(normals))

        # 复制入射方向向量，保持和原系统的对接语义
        d_hit = ray_d.clone()

        # =========================================================================
        # 6. 终极强类型维度护航（Shape Lockdown）
        # =========================================================================
        # 无论前面求交过程中因为掩码分流发生了何种退化，在 return 前强行使用 view() 锁死返回值的维度轴数
        d_hit = d_hit.view(R, 3)              # 雷打不动的二维入射方向张量 (R, 3)
        normals = normals.view(R, 3)          # 精确对应的单位法线张量 (R, 3)
        obj_ids = obj_ids.view(R, 2)          # 雷打不动的二维物性看板 (R, 2) -> 每行为 [接收方x, 发射方y]
        hit_points = hit_points.view(R, 3)    # 严格锁死的交点坐标 (R, 3)
        p_error = p_error.view(R, 3)          # 严格锁死的误差包络线 (R, 3)
        hit_mask = hit_mask.view(R)            # 稳固的一维布尔向量 (R,)
        min_tri_idx = min_tri_idx.view(R)      # 本次命中的真实 Tensor 行号位置，供外部状态机迭代交接

        # 将所有追迹所需的底层核心几何张量全异步打包交出
        return d_hit, normals, obj_ids, hit_points, p_error, hit_mask, min_tri_idx

# =====================================================================
# 3. PBRT 工业级多物理维度并行光追引擎核心类
# =====================================================================
class AdvancedRayTracerPBRTGPU:
    def __init__(self, x_lim, y_lim, z_lim, r_configs, a_configs, n_configs, scatter_configs=None,
                 scene_intersector=None, fresnel=True, default_medium=0, default_n=1.0,
                 div_epsilon=1e-9, front_epsilon=1e-5):
        """
        PBRT 工业级多物理维度并行光追引擎构造函数
        """
        # 将空间 X 轴的安全物理剪裁边界固化为 GPU 上的 Float32 张量
        self.x_lim = torch.tensor(x_lim, dtype=torch.float32, device=device)
        # 将空间 Y 轴的安全物理剪裁边界固化为 GPU 上的 Float32 张量
        self.y_lim = torch.tensor(y_lim, dtype=torch.float32, device=device)
        # 将空间 Z 轴的安全物理剪裁边界固化为 GPU 上的 Float32 张量
        self.z_lim = torch.tensor(z_lim, dtype=torch.float32, device=device)

        self.r_configs = r_configs            # 固化场景中各三角形物体的基础反射率（Reflectality）配置字典
        self.a_configs = a_configs            # 固化场景中各三角形物体的基础吸收率（Absorptivity）配置字典
        self.n_configs = n_configs            # 固化各物理介质的折射率（Refractive Index）映射配置字典（支持函数型色散）
        self.s_configs = scatter_configs if scatter_configs is not None else {} # 固化散射/漫反射比率配置，默认为空字典

        self.fresnel = fresnel                # 物理控开关：是否开启经典的菲涅尔（Fresnel）反射/透射能量分配效应
        self.default_medium = default_medium  # 定义场景的默认背景介质 ID（通常 0 代表真空或空气）
        self.default_n = default_n            # 定义场景背景介质的默认折射率基准值（真空下为 1.0）
        self.div_epsilon = div_epsilon        # 引擎内部通用除法防御极小量
        self.front_epsilon = front_epsilon    # 引擎内部通用的光线推进硬性安全偏置量

        # 若外部显式传入了实例化好的多边形求交引擎，则将其绑定到当前光追引擎中
        if scene_intersector is not None:
            self.scene_intersector = scene_intersector

    def _evaluate_property(self, obj_ids, nu, config_dict, default_val=0.0):
      """
      升级版物性评估函数：全面支持常数、解析函数（如 Cauchy 公式）以及离散采样表格（自动触发 gpu_interp1d）。
      """
      # 🟢 动态兼容：确保设备上下文与输入的 obj_ids 强一致，防止跨设备引发 RuntimeError
      current_device = obj_ids.device

      # 创建一个与当前物体ID数组等长、填满默认值的 Float32 张量，作为属性缓冲器
      res = torch.full_like(obj_ids, default_val, dtype=torch.float32, device=current_device)

      # 高效操作：利用 unique 提取当前碰撞涉及的所有不重复的物体 ID，避免无意义的遍历
      for oid in torch.unique(obj_ids):
          oid_item = oid.item()

          # 判断该物体 ID 是否注册在配置字典中
          if oid_item in config_dict:
              val = config_dict[oid_item]

              # 🟢 构造当前物体的局部布尔掩码，用于后续将插值或计算结果精准定向写入对应光线通道
              mask = (obj_ids == oid)

              # 提取当前物体对应的所有光线的物理频率/波长子集，作为插值查询点
              sub_nu = nu[mask] if isinstance(nu, torch.Tensor) else nu

              # 🟢 情况一：如果配置是可调用的（即为连续的色散/吸收曲线解析函数）
              if callable(val):
                  res = torch.where(mask, val(nu), res)

              # 🟢 情况二：如果配置是元组或列表，且长度为 2 (即代表输入的离散实验数据表格 [x_data, y_data])
              elif isinstance(val, (tuple, list)) and len(val) == 2:
                  x_data, y_data = val[0], val[1]

                  # 确保输入的离散表格数据已经转换为同一设备上的 PyTorch 张量
                  if not isinstance(x_data, torch.Tensor): x_data = torch.tensor(x_data, dtype=torch.float32, device=current_device)
                  if not isinstance(y_data, torch.Tensor): y_data = torch.tensor(y_data, dtype=torch.float32, device=current_device)

                  # 🚀 核心接入点：调用你编写的纯张量化 gpu_interp1d，为该物体的所有子光线并行插值解算出物理物性
                  interpolated_vals = gpu_interp1d(sub_nu, x_data, y_data)

                  # 利用高级索引或 where 算子，将插值出的精细物理量安全覆盖回对应光线的位置上
                  res[mask] = interpolated_vals

              # 🟢 情况三：如果配置是固定的常数数值
              else:
                  res = torch.where(mask, float(val), res)

      # 返回评估完物理属性（常数/函数/表格插值综合评估）后的最终张量
      return res

    def _sample_lanebertian_dir(self, normals):
        # 提取当前需要生成漫反射的光线总数 N
        N = normals.shape[0]
        # 在 [0, 1) 区间内并行生成第一个独立同分布的均匀随机数张量 u1，用于控制圆周方位角
        u1 = torch.rand(N, device=device)
        # 生成第二个独立同分布的均匀随机数张量 u2，用于实现余弦权重（Cosine-weighted）的天顶角采样
        u2 = torch.rand(N, device=device)

        # 方位角映射：将 u1 映射到 [0, 2π] 的完整几何圆周空间
        phi = 2.0 * torch.pi * u1
        # 根据麦夸特折射原理，天顶角的正弦分量 r 正好正比于 u2 的开平方
        r = torch.sqrt(u2)
        # 算出在切空间（Tangent Space）局部坐标系下的 X 轴分量坐标
        x = r * torch.cos(phi)
        # 算出在切空间局部坐标系下的 Y 轴分量坐标
        y = r * torch.sin(phi)
        # 算出在切空间局部坐标系下的 Z 轴分量坐标（严格垂直于本地表面平面），防溢出截断
        z = torch.sqrt(torch.clamp(1.0 - u2, min=0.0))

        # 将全局法向量定义为本地切空间的局部 Z 轴方向
        z_axis = normals
        # 构建一个世界坐标系下的基准辅助向量 [1, 0, 0]，并将其复制扩张到与法线张量相同的结构
        aux = torch.tensor([1.0, 0.0, 0.0], device=device).expand_like(z_axis)
        # 构建备用辅助向量 [0, 1, 0]，防止法线恰好与第一辅助向量平行而导致叉乘退化
        alt_aux = torch.tensor([0.0, 1.0, 0.0], device=device).expand_like(z_axis)

        # 健壮性自适应判定：如果法线与第一辅助向量的点积绝对值大于 0.5 则强制切换为备用辅助向量，否则维持主辅助向量
        use_alt = torch.abs(torch.sum(z_axis * aux, dim=-1, keepdim=True)) > 0.5
        up = torch.where(use_alt, alt_aux, aux)

        # 施密特正交化：通过辅助向量与 Z 轴叉乘，并行推导出切空间的局部 X 轴方向张量
        x_axis = torch.cross(up, z_axis, dim=-1)
        # 对局部 X 轴进行 L2 归一化，加上极小项防止分母为 0
        x_axis = x_axis / (torch.norm(x_axis, dim=-1, keepdim=True) + self.div_epsilon)
        # 右手坐标系法则：将已知的单位 Z 轴与单位 X 轴进行叉乘，直接导出正交的单位 Y 轴张量
        y_axis = torch.cross(z_axis, x_axis, dim=-1)

        # 基底线性组合：将切空间局部三维分量坐标 (x, y, z) 投影回 3D 全局世界坐标系，并返回新方向
        return x.unsqueeze(-1) * x_axis + y.unsqueeze(-1) * y_axis + z.unsqueeze(-1) * z_axis

    def _handle_advanced_physics(self, d_hit, normals, obj_ids, curr_medium_ids, nu, pol, p_error):
        """
        多物理维度核心算子：解算光流更新，并输出 PBRT 安全边界平移。
        """
        # 如果传入了偏振态张量，开始进行严密的维度自适应审查与补全
        if pol is not None:
            # 判定若偏振张量只有 1 维
            if pol.dim() == 1:
                # 若其长度恰好等同于光线数，说明是单通道标量，将其在末尾升维变成列向量
                if pol.shape[0] == d_hit.shape[0]: pol = pol.unsqueeze(-1)
                # 若其长度为 3 且当前光线数为 1，说明是单条光线的三维偏振电场矢量，将其在首部升维
                elif pol.shape[0] == 3 and d_hit.shape[0] == 1: pol = pol.unsqueeze(0)
            # 判定若偏振张量是 2 维且最后一维为 1
            if pol.dim() == 2 and pol.shape[-1] == 1:
                # 属于强度标量，直接复制扩张为 3D 电场振幅矢量形式，以支持复杂的偏振矩阵运算
                pol = pol.expand(-1, 3)

        # 提取当前参与物理作用的光线总数 N
        N = len(obj_ids)

        # 列表推导结合张量化：批量收集场景中定义的三角形面片的散射/粗糙漫反射率
        # s_rate = torch.tensor([self.s_configs.get(oid.item(), 0.0) for oid in obj_ids], dtype=torch.float32, device=device)
        # 动态查表：解算出当前光线在宿主空间中所处前续介质的真实折射率 n_curr
        # n_curr = self._evaluate_property(curr_medium_ids, nu, self.n_configs, default_val=self.default_n)

        # 1. 批量提取命中面片两侧的物体ID：[Sub_Batch, 2]
        # hit_obj_ids 此时由单列变为双列
        hit_obj_ids = obj_ids

        # 2. 提取法线外侧(0)与内侧(1)的真实物体类别
        obj_id_outer = hit_obj_ids[:, 0]
        obj_id_inner = hit_obj_ids[:, 1]

        # 3. 动态查表，并行获取两侧介质在当前频率下的折射率
        n_outer = self._evaluate_property(obj_id_outer, nu, self.n_configs, default_val=1.0)
        n_inner = self._evaluate_property(obj_id_inner, nu, self.n_configs, default_val=1.0)

        # 查出外侧和内侧各自的反射率基准
        r_base_outer = self._evaluate_property(obj_id_outer, nu, self.r_configs, default_val=0.0)
        r_base_inner = self._evaluate_property(obj_id_inner, nu, self.r_configs, default_val=0.0)

        # 查出外侧和内侧各自的吸收率基准
        a_base_outer = self._evaluate_property(obj_id_outer, nu, self.a_configs, default_val=0.0)
        a_base_inner = self._evaluate_property(obj_id_inner, nu, self.a_configs, default_val=0.0)

        # 列表推导结合张量化：批量收集场景中定义的三角形面片的散射/粗糙漫反射率
        s_rate_outer = self._evaluate_property(obj_id_outer, nu, self.s_configs, default_val=0.0)
        s_rate_inner = self._evaluate_property(obj_id_inner, nu, self.s_configs, default_val=0.0)

        # 点积计算：计算入射光线方向与几何面片法线的点积，决定光线入射角度
        dot_in = torch.sum(d_hit * normals, dim=-1)
        # 布尔掩码：点积小于 0 说明光线从外部射入物体内部；点积大于 0 说明光线从物体内部准备射出外部
        entering = dot_in < 0

        # 法线自适应纠正：如果光线射出物体，需要将几何法线强行反转，确保法线始终迎着入射光线的方向
        corrected_normals = torch.where(entering.unsqueeze(-1), normals, -normals)
        # 根据几何几何关系，入射角的余弦值 cos_i 等于该点积的绝对值
        cos_i = torch.abs(dot_in)
        # 介质判定：若射入物体，则第一介质折射率 n1 = 当前介质，第二介质 n2 = 目标物体折射率
        # n1 = torch.where(entering, n_curr, n_trans)
        # 反之若射出物体，则第一介质 n1 = 物体折射率，第二介质 n2 = 外部当前环境折射率
        # n2 = torch.where(entering, n_trans, n_curr)

        # 4. 根据光线入射的几何朝向（entering掩码）动态赋予第一(n1)和第二(n2)介质折射率
        # entering = True (点积<0) -> 说明光线从面片法线外侧撞击，准备切入内侧

        # 动态查表：并行获取被击中物体的固有材质反射率基准张量 r_base, a_base
        r_base = torch.where(entering, r_base_inner, r_base_outer)
        a_base = torch.where(entering, a_base_inner, a_base_outer)
        s_rate = torch.where(entering, s_rate_inner, s_rate_outer)

        n1 = torch.where(entering, n_outer, n_inner) # 入射侧折射率
        n2 = torch.where(entering, n_inner, n_outer) # 透射侧折射率

        # 计算两种介质的相对折射率比值 η (eta)
        eta = n1 / n2

        # 根据斯涅尔定律（Snell's Law），计算透射角的正弦平方值 sin²(t) = η² * (1 - cos²(i))
        sin2_t = eta**2 * (1.0 - cos_i**2)
        # 全反射判定掩码：如果透射角正弦平方值大于 1.0，说明发生全反射（Total Internal Reflection）
        total_internal_reflection = sin2_t > 1.0
        # 根据三角恒等式解出透射角余弦值 cos_t，利用 clamp 截断任何由浮点数舍入带来的微小负数
        cos_t = torch.sqrt(torch.clamp(1.0 - sin2_t, min=0.0))

        # 经典菲涅尔方程：计算 P 偏振分量的反射电场振幅（振幅反射率）
        r_p_amp = (n2 * cos_i - n1 * cos_t) / (n2 * cos_i + n1 * cos_t + self.div_epsilon)
        # 经典菲涅尔方程：计算 S 偏振分量的反射电场振幅（振幅反射率）
        r_s_amp = (n1 * cos_i - n2 * cos_t) / (n1 * cos_i + n2 * cos_t + self.div_epsilon)

        # 根据物理光学，能量反射率等于振幅反射率的平方，解出 P 偏振波的功率反射率
        fresnel_p = r_p_amp**2
        # 解出 S 偏振波的功率反射率
        fresnel_s = r_s_amp**2

        # 经典菲涅尔方程：解出 P 偏振分量的透射电场振幅
        t_p_amp = (2.0 * n1 * cos_i) / (n2 * cos_i + n1 * cos_t + self.div_epsilon)
        # 经典菲涅尔方程：解出 S 偏振分量的透射电场振幅
        t_s_amp = (2.0 * n1 * cos_i) / (n1 * cos_i + n2 * cos_t + self.div_epsilon)

        # 条件物理大分支：解算最终的能量分配系数
        if not self.fresnel:
            # 1. 若关闭菲涅尔效应，反射因子强行恒置为 1.0（由材质固有配置主导）
            fresnel_factor = torch.ones_like(fresnel_p)
            S_dir, E_s_amp, E_p_amp = None, None, None
        elif pol is None:
            # 2. 若开启菲涅尔但无偏振态输入，退化为无偏振自然光，反射率取 S 与 P 偏振功率反射率的几何算术平均值
            fresnel_factor = 0.5 * (fresnel_p + fresnel_s)
            S_dir, E_s_amp, E_p_amp = None, None, None
        else:
            # 3. 🟢 【高级物理】光线携带有偏振态：必须进行严密的偏振坐标系分解
            # 叉乘求出垂直于入射面（由入射方向和法线构成）的 S 偏振基向量 S_dir
            S_dir = torch.cross(d_hit, corrected_normals, dim=-1)
            # 对 S 偏振方向基矢进行严格单位化
            S_dir = S_dir / (torch.norm(S_dir, dim=-1, keepdim=True) + self.div_epsilon)

            # 叉乘求出平行于入射面的 P 偏振基向量 P_dir_in
            P_dir_in = torch.cross(S_dir, d_hit, dim=-1)
            # 对 P 偏振方向基矢进行严格单位化
            P_dir_in = P_dir_in / (torch.norm(P_dir_in, dim=-1, keepdim=True) + self.div_epsilon)

            # 将输入的电场矢量投影到 S 基矢上，得到 S 偏振分量的当前复振幅 E_s_amp
            E_s_amp = torch.sum(pol * S_dir, dim=-1, keepdim=True)
            # 将输入的电场矢量投影到 P 基矢上，得到 P 偏振分量的当前复振幅 E_p_amp
            E_p_amp = torch.sum(pol * P_dir_in, dim=-1, keepdim=True)

            # 依据玻印亭矢量能量守恒，复合计算出当前明确偏振状态下的真实综合菲涅尔功率反射因子
            fresnel_factor = (E_s_amp**2 * fresnel_s.unsqueeze(-1) + E_p_amp**2 * fresnel_p.unsqueeze(-1)) / (E_s_amp**2 + E_p_amp**2 + self.div_epsilon)
            # 将多余的尾部单一维度挤压消除
            fresnel_factor = fresnel_factor.squeeze(-1)

        # 物理修正：如果掩码指示发生了全反射，反射能量比例无条件强行置为 1.0（能量 100% 反射）
        fresnel_factor = torch.where(total_internal_reflection, torch.ones_like(fresnel_factor), fresnel_factor)
        print('total_internal_reflection:', total_internal_reflection) # debug

        # 最终综合物理反射率 = 材质基础反射率 × 菲涅尔修正因子
        r_actual = r_base * fresnel_factor
        # 最终综合物理吸收率 = 材质固有吸收率
        a_actual = a_base
        # 能量守恒定律：透射率 t = 1.0 - r - a，并用 clamp 确保不为负数
        t_actual = torch.clamp(1.0 - r_actual - a_actual, min=0.0)

        # 计算总概率权重和，作为概率归一化分母
        total_prob = r_actual + a_actual + t_actual + self.div_epsilon
        # 蒙特卡洛生存轮盘赌：归一化后的光线在当前界面发生反射的概率
        p_reflect = r_actual / total_prob
        # 归一化后的光线在当前界面被吸收的概率
        p_absorb = a_actual / total_prob

        # 蒙特卡洛并行采样：为每条光线生成一个 [0, 1) 的随机浮点数作为抉择指针
        rand_choice = torch.rand(N, device=device)
        # 判定发生反射的命运掩码（落在反射概率区间内）
        reflect_mask = rand_choice < p_reflect
        # 判定发生吸收的命运掩码（落在紧随其后的吸收概率区间内）
        absorb_mask = (rand_choice >= p_reflect) & (rand_choice < (p_reflect + p_absorb))
        # 命运互斥：既未反射也未吸收，则必然归于透射掩码
        transmit_mask = ~(reflect_mask | absorb_mask)

        # 物理约束：如果发生了全反射，透射通道强行关闭
        transmit_mask = transmit_mask & (~total_internal_reflection)
        # 如果发生了全反射，该光线命运强行修改为归属于反射通道
        reflect_mask = torch.where(total_internal_reflection, torch.ones_like(reflect_mask), reflect_mask)

        # 再生成一组相互独立的随机数，用于在反射内部抉择是镜面反射还是漫反射散射
        rand_scatter = torch.rand(N, device=device)
        # 漫反射判定掩码：处于反射命运中，且随机数小于材质固有的散射率 s_rate
        diffuse_mask = reflect_mask & (rand_scatter < s_rate)
        # 镜面反射判定掩码：处于反射命运中，但不满足漫反射条件的光线
        specular_mask = reflect_mask & (~diffuse_mask)

        # 克隆一份初始方向张量作为光流更新缓冲器
        new_d = d_hit.clone()
        # 克隆一份当前的介质ID张量作为空间跨越缓冲器
        new_medium_ids = curr_medium_ids.clone()
        #new_medium_ids = torch.where(entering, obj_id_inner, obj_id_outer)

        # 经典光学公式：计算完美的理想镜面反射方向：D_out = D_in - 2 * (D_in · N) * N
        reflect_d = d_hit - 2.0 * dot_in.unsqueeze(-1) * normals

        # 如果有光线命中了镜面反射通路
        if specular_mask.any():
            print('reflection') # debug
            # 抓取对应光线在当前局部批次中的索引序号
            spec_indices = torch.where(specular_mask)[0]
            # 批量把计算好的理想镜面反射矢量写入对应通道
            new_d[spec_indices, :] = reflect_d[spec_indices, :]

        # 如果有光线命中了漫反射通路
        if diffuse_mask.any():
            print('diffuse reflection') # debug
            # 抓取漫反射光线的对应索引
            diff_indices = torch.where(diffuse_mask)[0]
            # 调用余弦权重半球采样函数，为其并行赋上符合朗伯特（Lambertian）余弦定律的全新散射方向
            new_d[diff_indices, :] = self._sample_lanebertian_dir(corrected_normals[diff_indices])

        # 如果有光线命中了透射/折射通路
        if transmit_mask.any():
            print('refraction') # debug
            # 抓取透射光线的对应索引
            trans_indices = torch.where(transmit_mask)[0]
            # 提取这些透射光线专属的相对折射率比值分量
            eta_m = eta.view(-1)[trans_indices].unsqueeze(-1)
            # 提取入射角余弦分量
            cos_i_m = cos_i.view(-1)[trans_indices].unsqueeze(-1)
            # 提取透射角余弦分量
            cos_t_m = cos_t.view(-1)[trans_indices].unsqueeze(-1)
            # 提取对应的入射方向矢量
            d_hit_m = d_hit[trans_indices]
            # 提取校正后的法线矢量
            norm_m = corrected_normals[trans_indices]

            # 严密三维斯涅尔折射方程：D_refract = η * D_in + (η * cos(i) - cos(t)) * N
            refract_d = eta_m * d_hit_m + (eta_m * cos_i_m - cos_t_m) * norm_m
            # 归一化折射矢量并强力填入新方向缓冲
            new_d[trans_indices, :] = refract_d / (torch.norm(refract_d, dim=-1, keepdim=True) + self.div_epsilon)
            # 🟢 空间介质状态跃迁：如果射入物体(entering为True)，介质ID改为被击中物体ID，否则退回原ids
            # new_medium_ids[trans_indices] = torch.where(entering[trans_indices], obj_ids[trans_indices], new_medium_ids[trans_indices])

            # =========================================================================
            # 🟢 空间介质状态跃迁：零 IF 分支、全张量化剥离双列材质看板
            # =========================================================================
            # 1. 直接剥离全场所有击中光线对应的外侧和内侧物体ID
            obj_id_outer_all = obj_ids[:, 0] # 这两行是否多余？
            obj_id_inner_all = obj_ids[:, 1]

            # 2. 依据进出状态决定物理期望新介质：射入(True)->内侧物体ID； 射出(False)->外侧物体ID
            intended_medium_ids = torch.where(entering, obj_id_inner_all, obj_id_outer_all)

            # 🚀 3. 终极数值防御：将整数索引/掩码安全广播恢复为与全场对齐的纯布尔开关 (True/False)
            # 创建一个与 new_medium_ids 等长的全 False 布尔看板
            trans_bool_mask = torch.zeros_like(new_medium_ids, dtype=torch.bool)
            # 将发生透射的坑位强制点亮为 True
            trans_bool_mask[trans_indices] = True

            # 4. 此时 trans_bool_mask 已经是标准的布尔张量，完美通过 PyTorch 的强类型检查
            new_medium_ids = torch.where(trans_bool_mask, intended_medium_ids, new_medium_ids)

        # 如果光线不幸被物理吸收
        if absorb_mask.any():
            print('absorption') # debug
            # 抓取被吸收光线的对应索引
            abs_indices = torch.where(absorb_mask)[0]
            # 强制将其运动方向矢量清零，终止其在世界空间中的继续传播
            new_d[abs_indices, :] = 0.0

        # =====================================================================
        # 🟢 【PBRT 工业级核心修改 2】：自交伪影平移向量重构
        # =====================================================================
        # PBRT 的规范做法：各个坐标轴的误差是分立的，偏移方向直接取决于法线分量的正负号。
        # 如果 normal > 0，在该轴向外推进 p_error；反之向负方向扣除。
        sign_direction = torch.where(corrected_normals > 0, torch.ones_like(corrected_normals), -torch.ones_like(corrected_normals))
        # 各轴独立计算出绝对安全的浮点数错位偏置矢量
        pbrt_offset = sign_direction * p_error

        # 核心逻辑：透射光进入物体内部，应沿法线负方向扣除偏置；反射光向外空间扩散，应沿法线正方向累加偏置
        bias_vector = torch.where(transmit_mask.unsqueeze(-1), -pbrt_offset, pbrt_offset)

        # =====================================================================
        # 偏振态状态转移矩阵结算
        # =====================================================================
        # 如果存在偏振振幅，则深度克隆一份副本作为电场演化缓冲器
        new_pol = pol.view(N, 3).clone() if pol is not None else None

        # 判定若存在需要演化的偏振电场矢量
        if pol is not None and S_dir is not None:
            # 1. 对镜面反射通道的光线结算偏振电场的菲涅尔衰减与相位跃迁
            if specular_mask.any():
                spec_global_idx = torch.where(specular_mask)[0]
                S_dir_m = S_dir[spec_global_idx]
                new_d_m = new_d[spec_global_idx]
                E_s_amp_m = E_s_amp[spec_global_idx]
                E_p_amp_m = E_p_amp[spec_global_idx]
                r_s_amp_m = r_s_amp[spec_global_idx].unsqueeze(-1)
                r_p_amp_m = r_p_amp[spec_global_idx].unsqueeze(-1)

                # 反射电场更新：计算反射后的全新 P 偏振方向（垂直于反射光和 S 轴）
                P_dir_reflect = torch.cross(S_dir_m, new_d_m, dim=-1)
                P_dir_reflect = P_dir_reflect / (torch.norm(P_dir_reflect, dim=-1, keepdim=True) + self.div_epsilon)
                # 叠加原理：更新后的总反射偏振电场矢量 E_new = E_s * r_s * S_dir + E_p * r_p * P_dir
                new_pol[spec_global_idx, :] = E_s_amp_m * r_s_amp_m * S_dir_m + E_p_amp_m * r_p_amp_m * P_dir_reflect

            # 2. 对粗糙度漫反射通道的光线进行偏振态退偏（消偏振化）非相干处理
            if diffuse_mask.any():
                diff_global_idx = torch.where(diffuse_mask)[0]
                diffuse_d = new_d[diff_global_idx]
                aux = torch.tensor([1.0, 0.0, 0.0], device=device).expand_like(diffuse_d)
                alt_aux = torch.tensor([0.0, 1.0, 0.0], device=device).expand_like(diffuse_d)
                use_alt = torch.abs(torch.sum(diffuse_d * aux, dim=-1, keepdim=True)) > 0.5
                rand_axis = torch.where(use_alt, alt_aux, aux)

                # 机制：由于无序多重散射，漫反射输出的波退化为完全随机偏振光（随机生成一个偏振电场基矢）
                random_E = torch.cross(diffuse_d, rand_axis, dim=-1)
                new_pol[diff_global_idx, :] = random_E / (torch.norm(random_E, dim=-1, keepdim=True) + self.div_epsilon)

            # 3. 对透射折射通道的光线结算透射偏振电场的演化
            if transmit_mask.any():
                trans_global_idx = torch.where(transmit_mask)[0]
                S_dir_m = S_dir[trans_global_idx]
                new_d_m = new_d[trans_global_idx]
                E_s_amp_m = E_s_amp[trans_global_idx]
                E_p_amp_m = E_p_amp[trans_global_idx]
                t_s_amp_m = t_s_amp[trans_global_idx].unsqueeze(-1)
                t_p_amp_m = t_p_amp[trans_global_idx].unsqueeze(-1)

                # 透射电场更新：计算透射后的全新 P 偏振基矢方向
                P_dir_transmit = torch.cross(S_dir_m, new_d_m, dim=-1)
                P_dir_transmit = P_dir_transmit / (torch.norm(P_dir_transmit, dim=-1, keepdim=True) + self.div_epsilon)
                # 叠加原理：更新后的总透射偏振电场矢量 E_new = E_s * t_s * S_dir + E_p * t_p * P_dir
                new_pol[trans_global_idx, :] = E_s_amp_m * t_s_amp_m * S_dir_m + E_p_amp_m * t_p_amp_m * P_dir_transmit

            # 偏振能量守恒重归一化：保证未被吸收的活动光线偏振波其电场模长依然与光强对应
            active_mask = ~absorb_mask
            if active_mask.any():
                active_global_idx = torch.where(active_mask)[0]
                new_pol[active_global_idx, :] = new_pol[active_global_idx, :]/ (torch.norm(new_pol[active_global_idx, :], dim=-1, keepdim=True) + self.div_epsilon)

        # 🟢 调整返回值，直接返回解算好的分立轴向 3D 安全偏置向量（维度 [N, 3]）以及各项物理状态
        return new_d, new_medium_ids, reflect_mask, transmit_mask, absorb_mask, bias_vector, new_pol

    def trace(self, origins_np, directions_np, nu_np, polarization_np=None, max_bounces=12):
        """
        零内层循环全局光线追踪主控制演化函数。
        已集成：基于 Tensor 保存位置的自相交拓扑级免疫机制。
        """

        # 自动化自检：防止用户忘记在类初始化时绑定场景几何求交引擎
        if not hasattr(self, 'scene_intersector'):
            global_context = globals()
            found_intersector = None
            # 在外部全局命名空间中自动搜寻潜在的求交引擎变量名
            for candidate in ['intersector', 'scene_intersector', 'mesh_intersector', 'bvh_engine']:
                if candidate in global_context:
                    found_intersector = global_context[candidate]
                    break

            # 若搜寻成功，实施智能拦截和动态向内绑定，确保引擎不中断崩溃
            if found_intersector is not None:
                self.scene_intersector = found_intersector
                print(f"   [🔥 智能拦截] 检测到类内部未绑定 scene_intersector，已自动动态绑定外部对象: '{candidate}'")
            else:
                # 搜寻失败则抛出严重异常
                raise ValueError("AdvancedRayTracerPBRTGPU 缺失核心求交引擎 'scene_intersector'。")

        # 数据上机：将传入的起点 NumPy 数组转换为计算设备上的 PyTorch 张量
        origins = torch.tensor(origins_np, dtype=torch.float32, device=device)
        # 将传入的方向矢量 NumPy 数组转换为显存中的张量
        directions = torch.tensor(directions_np, dtype=torch.float32, device=device)
        # 将各光线特有的物理频率/波长转换为显存张量
        nu = torch.tensor(nu_np, dtype=torch.float32, device=device)

        pol = None
        # 如果用户显式指定了初始光线的偏振态数组，进行严密转换与形状对齐
        if polarization_np is not None:
            pol_raw = torch.tensor(polarization_np, dtype=torch.float32, device=device)
            if pol_raw.dim() == 1:
                pol = pol_raw.view(-1, 3).clone()
            elif pol_raw.dim() == 2 and pol_raw.shape[-1] == 1:
                pol = pol_raw.expand(-1, 3).contiguous()
            else:
                pol = pol_raw.contiguous()

        # 统计本批次并发发射击的全局总光线数
        N_total = origins.shape[0]

        # 状态看板跟踪：0 代表活动中(Active)；1 代表被材质吸收(Absorbed)；2 代表飞向宇宙深处未击中几何体(Escaped)
        ray_status = torch.zeros(N_total, dtype=torch.long, device=device)
        # 介质追踪器：初始化所有光线的前续宿主空间介质皆为默认背景(0)
        curr_medium_ids = torch.zeros(N_total, dtype=torch.long, device=device)

        # 🚀 核心新增 1：初始化全局所有光线的前次命中三角形的 Tensor 位置记录看板
        # 全部填 -1，代表初始状态下谁也没有上一轮的相交历史
        global_prev_hit_face_ids = torch.full((N_total,), -1, dtype=torch.int64, device=device)

        # 创建一个空列表，用于冷冻记录每一弹跳步数（Bounce Step）时全场光线的几何与物理快照
        ray_history = []
        # 将初始发射状态作为第 0 步快照强行压入历史栈中
        ray_history.append({
            'origins': origins.clone(),
            'directions': directions.clone(),
            'medium_ids': curr_medium_ids.clone(),
            'polarization': pol.clone() if pol is not None else None
        })

        # 核心外层时间演化循环（唯一允许的显式循环，控制总弹跳深度）
        for bounce in range(max_bounces):
            # 并行创建活动光线掩码：只有状态仍为 0 的光线才有资格继续传播
            active_mask = (ray_status == 0)

            # 提前收敛优化：如果全场已经没有一条光线是活动的，直接跳出循环，节省巨量无效计算开销
            if not active_mask.any():
                print(f"   [弹跳中断] 第 {bounce} 步：全场活动光线已清零，演化提前收敛。")
                break

            # 核心张量压缩：提取全场所有活动光线在原始大数组里的全局索引序号
            active_global_indices = torch.where(active_mask)[0]
            # 抽取活动光线的起点子张量
            sub_origins = origins[active_global_indices]
            # 抽取活动光线的方向子张量
            sub_directions = directions[active_global_indices]
            # 抽取活动光线的物理频率子张量
            sub_nu = nu[active_global_indices]
            # 抽取活动光线当前的宿主介质子张量
            sub_medium_ids = curr_medium_ids[active_global_indices]
            # 抽取活动光线的当前偏振态子张量
            sub_pol = pol[active_global_indices, :] if pol is not None else None

            # 🚀 核心新增 2：同步切片提取这些仍在活动的光线各自的“前一次命中面位置历史”
            sub_prev_hit_face_ids = global_prev_hit_face_ids[active_global_indices]

            # 🛠 核心调用：将压缩后的活动数据及前任行号历史灌入新版求交引擎
            # 增加接收：接收底层过滤返回的 hit_mask_sub(本轮真实命中) 和 min_tri_idx(本轮命中的新行号位置)
            d_hit, normals, obj_ids, hit_positions_sub, p_error, hit_mask_sub, min_tri_idx = \
                self.scene_intersector.intersect_batch(sub_origins, sub_directions, sub_prev_hit_face_ids)

            # 子集未命中掩码：通过取反高并行的 hit_mask_sub 完美获得，彻底告别原代码的 (obj_ids < 0) 膨胀隐患
            miss_mask_sub = ~hit_mask_sub

            # 如果活动子集中有光线射向无穷远
            if miss_mask_sub.any():
                # 通过双重映射，把子集里未命中光线的局部序号还原为全局大数组中的绝对索引
                miss_idx_global = active_global_indices[torch.where(miss_mask_sub)[0]]

                esc_o = origins[miss_idx_global, :]
                esc_d = directions[miss_idx_global, :]

                # 组装宇宙边界
                box_min = torch.tensor([self.x_lim[0], self.y_lim[0], self.z_lim[0]], device=device, dtype=esc_o.dtype)
                box_max = torch.tensor([self.x_lim[1], self.y_lim[1], self.z_lim[1]], device=device, dtype=esc_o.dtype)

                # 🟢 【工业级标准 AABB 防死区求交算法】
                inv_d = 1.0 / (esc_d + torch.sign(esc_d + 1e-12) * 1e-6)

                t1 = (box_min - esc_o) * inv_d
                t2 = (box_max - esc_o) * inv_d

                # 严格区分进入点(t_near)和离开点(t_far)
                t_near = torch.minimum(t1, t2)
                t_far = torch.maximum(t1, t2)

                # 物理准则：在所有轴向中，最晚进入包围盒（max）且最早离开包围盒（min）的距离
                t_hit_start = torch.max(t_near, dim=-1)[0]
                t_hit_end = torch.min(t_far, dim=-1)[0]

                # 如果光线起点就在盒子内部，真正向前看撞击到外壁的距离是 t_hit_end
                t_hit = torch.where(t_hit_start > 0, t_hit_start, t_hit_end)
                t_hit = torch.clamp(t_hit, min=1e-3)

                # 计算外围大盒子上的确切世界坐标 [M, 3]
                boundary_hit_points = esc_o + t_hit.unsqueeze(-1) * esc_d

                # 强制回填全局看板
                origins[miss_idx_global, :] = boundary_hit_points
                directions[miss_idx_global, :] = 0.0 # 冻结能量，退出后续追迹
                ray_status[miss_idx_global] = 2

                # 🚀 核心新增 3：飞向深空的光线，其前次碰撞记录可以安全重置为 -1
                global_prev_hit_face_ids[miss_idx_global] = -1

            # 如果活动子集中有光线成功命中场景几何面片
            if hit_mask_sub.any():
                # 还原出这些真正命中的光线在全局大数组中的绝对索引
                hit_idx_global = active_global_indices[torch.where(hit_mask_sub)[0]]

                # 对解算出的底层几何量进行二次筛选提取
                d_hit_h = d_hit[hit_mask_sub]
                normals_h = normals[hit_mask_sub]
                obj_ids_h = obj_ids[hit_mask_sub, :]
                curr_medium_ids_h = sub_medium_ids[hit_mask_sub]
                nu_h = sub_nu[hit_mask_sub]
                pol_h = sub_pol[hit_mask_sub, :] if sub_pol is not None else None
                p_error_h = p_error[hit_mask_sub]
                directions_h = sub_directions[hit_mask_sub]
                hit_positions_h = hit_positions_sub[hit_mask_sub]

                # debug
                print('d_hit_h:', d_hit_h)
                print('normals_h:', normals_h)
                print('obj_ids_h:', obj_ids_h)
                print('curr_medium_ids_h:', curr_medium_ids_h)
                print('nu_h:', nu_h)
                print('pol_h:', pol_h)
                print('p_error_h:', p_error_h)
                print('directions_h:', directions_h)
                #####################################################

                # 🚀 核心新增 4：【状态交接】
                # 将本次碰撞的真实新行号位置（min_tri_idx[hit_mask_sub]）通过全局绝对索引精准写入大记录看板。
                # 这样在下一轮弹跳迭代（bounce + 1）时，这批光线在底层求交时会自动对这一行面片产生免疫！
                # 而对于再下一次弹跳（bounce + 2），如果它们撞到了别的面，此行记录就会被新行号覆盖，原来的面自动重获计算合规性。
                global_prev_hit_face_ids[hit_idx_global] = min_tri_idx[hit_mask_sub]

                # 🛠 【PBRT 工业级核心修改 3】：将精细提取的几何撞击子集送入高级多维物理处理器
                new_d_h, new_med_h, reflect_mask_h, transmit_mask_h, absorb_mask_h, bias_vector_h, new_pol_h = \
                    self._handle_advanced_physics(
                        directions_h, normals_h, obj_ids_h, curr_medium_ids_h, nu_h, pol_h, p_error_h
                    )

                # debug
                print('new_d_h:', new_d_h)
                print('new_med_h:', new_med_h)
                print('reflect_mask_h:', reflect_mask_h)
                print('transmit_mask_h:', transmit_mask_h)
                print('absorb_mask_h:', absorb_mask_h)
                print('bias_vector_h:', bias_vector_h)
                print('new_pol_h:', new_pol_h)
                #################################################

                # 如果存在偏振态，将物理算子结算出的最新偏振电场矢量通过全局索引精准覆盖更新回去
                if pol is not None and new_pol_h is not None:
                    pol[hit_idx_global, :] = new_pol_h

                # 如果命运轮盘赌判定有一部分光线被吸收了
                if absorb_mask_h.any():
                    print('absorbed') # debug
                    # 1. 搜寻还原这些被吸收光线在全局数组中的绝对索引
                    absorb_idx_global = hit_idx_global[torch.where(absorb_mask_h)[0]]

                    # 2. 在全局生命看板上将其生命状态标记为 1 (Absorbed 被吸收)
                    ray_status[absorb_idx_global] = 1

                    # 🚀 3. 【核心修复】将这批光线的全局起点强行同步为精确的几何碰撞点
                    # 剥离出被吸收光线对应的物理交点坐标 [数量, 3]
                    origins[absorb_idx_global, :] = hit_positions_h[absorb_mask_h]

                    # 🚀 4. 【核心修复】将这批光线的全局方向冻结清零（用于终止后续历史轨迹画线的延伸）
                    # 这样本步历史记录中，它们的方向变为了 0，可视化线条就会在这里优雅地戛然而止
                    directions[absorb_idx_global, :] = 0.0

                    # 5. 被吸收的光线后续生命终结，重置其碰撞前任标签
                    global_prev_hit_face_ids[absorb_idx_global] = -1

                # 收集反射与透射光线，它们是能在本次弹跳中幸存并继续增殖演化的活动光线
                survived_mask_h = reflect_mask_h | transmit_mask_h
                # 如果确实有光线有幸存活
                if survived_mask_h.any():
                    # 映射还原这些幸存光线在全局大数组中的绝对索引
                    survived_idx_global = hit_idx_global[torch.where(survived_mask_h)[0]]
                    target_fill_count = survived_idx_global.shape[0] # 本轮主循环期望回填的逻辑粒子数

                    # =====================================================================
                    # 🚀 工业级安全分治回填舱（彻底隔离张量，严防二次维度裂变）
                    # =====================================================================

                    # ---- [ 1. 动态更新：方向矢量 ] ----
                    actual_spatial_rays = new_d_h.shape[0]
                    if actual_spatial_rays != target_fill_count:
                        if target_fill_count % actual_spatial_rays == 0:
                            new_d_h_aligned = new_d_h.repeat_interleave(target_fill_count // actual_spatial_rays, dim=0)
                        else:
                            raise RuntimeError(f"无法动态对齐方向维度！坑位: {target_fill_count}, 输出: {actual_spatial_rays}")
                    else:
                        new_d_h_aligned = new_d_h

                    directions[survived_idx_global, :] = new_d_h_aligned


                    # ---- [ 2. 动态更新：介质状态 ID ] ----
                    new_med_h = new_med_h.view(-1)
                    actual_med_count = new_med_h.shape[0]
                    if actual_med_count != target_fill_count:
                        if target_fill_count % actual_med_count == 0:
                            new_med_h_aligned = new_med_h.repeat_interleave(target_fill_count // actual_med_count, dim=0)
                        else:
                            new_med_h_aligned = new_med_h
                    else:
                        new_med_h_aligned = new_med_h

                    curr_medium_ids[survived_idx_global] = new_med_h_aligned


                    # ---- [ 3. 动态更新：起点坐标 ] ----
                    hit_positions_h = hit_positions_h.view(-1, 3)
                    bias_vector_h = bias_vector_h.view(-1, 3)

                    # A. 独立对齐物理交点坐标
                    pos_count = hit_positions_h.shape[0]
                    if pos_count != target_fill_count:
                        if target_fill_count % pos_count == 0:
                            hit_positions_aligned = hit_positions_h.repeat_interleave(target_fill_count // pos_count, dim=0)
                        else:
                            raise RuntimeError(f"交点坐标无法整除对齐！期望: {target_fill_count}, 实际: {pos_count}")
                    else:
                        hit_positions_aligned = hit_positions_h

                    # B. 独立对齐 PBRT 避自交偏置向量
                    bias_count = bias_vector_h.shape[0]
                    if bias_count != target_fill_count:
                        if target_fill_count % bias_count == 0:
                            bias_vector_aligned = bias_vector_h.repeat_interleave(target_fill_count // bias_count, dim=0)
                        else:
                            raise RuntimeError(f"偏置向量无法整除对齐！期望: {target_fill_count}, 实际: {bias_count}")
                    else:
                        bias_vector_aligned = bias_vector_h

                    # 🟢 点对点安全求和填入新起点
                    origins[survived_idx_global, :] = hit_positions_aligned + bias_vector_aligned

            # 将当前弹跳步骤演化完成后、全场所有光线的最新几何分布与状态打包存入历史记录
            ray_history.append({
                'origins': origins.clone(),
                'directions': directions.clone(),
                'medium_ids': curr_medium_ids.clone(),
                'polarization': pol.clone() if pol is not None else None
            })

        # 演化大循环全面结束，向外部调用者返回完整的光线追迹演化轨迹历史以及最终的生命看板状态
        return ray_history, ray_status




核心物理计算引擎已绑定至设备: cuda


In [ ]:
# generate rays: n = 2: Brewster's angle: 63.435 degrees
ray_orig, ray_dir, ray_frq, ray_pol = generate_rays(num_rays = 1,direction_angles = [63.435, 0], radius = 10, z_start = -30, polarization = 's')

In [ ]:
print(ray_orig)
print(ray_dir)
print(ray_frq)
print(ray_pol)

[[ -4.472128 -10.       -30.      ]]
[[0.8944276 0.        0.4472128]]
[[5.e+14]]
[[ 0. -1.  0.]]


测试正确以上code正确

给我写个函数 输入是dict：

{物体名字： stl文件path， ...}

输出是可以feed给SimpleMeshIntersector()的张量

In [ ]:
# vertices, faces, obj_ids: 这些接受merge_coplanar_meshes()输出 # resume here
# 1. 一键加载 .pt 文件（如果你是在 GPU 上保存的，建议加上 map_location='cpu' 确保安全）
loaded_data = torch.load("merged_mesh_data.pt", map_location=torch.device(device))

# 2. 从字典中根据 Key 把各个张量提取出来
vertices = loaded_data['vertices']
faces = loaded_data['faces']
obj_ids = loaded_data['labels'] # 这里第一个物体id是0 -1是真空 所以每个id += 1 因为光学engine 0是真空 正数代表每个物体
obj_ids += 1 # In optical engine, object index starts from 1 and 0 stands for external medium (vacuum)

# 3. 🟢 直接 feed 给你的 SimpleMeshIntersector
intersector = SimpleMeshIntersector(vertices, faces, obj_ids)

In [ ]:
print('vertices: ',vertices)
print('faces: ',faces)
print('obj_ids: ',obj_ids)
print('vertices shape: ',vertices.shape)
print('faces shape: ',faces.shape)
print('obj_ids shape: ',obj_ids.shape)

vertices:  tensor([[-2.0000e+01, -2.0000e+01, -2.0000e+01],
        [-2.0000e+01, -2.0000e+01,  2.0000e+01],
        [-2.0000e+01, -1.4000e-04,  2.0000e+01],
        [-2.0000e+01,  1.4000e-04,  2.0000e+01],
        [-2.0000e+01,  2.0000e+01, -2.0000e+01],
        [-2.0000e+01,  2.0000e+01,  2.0000e+01],
        [-1.0000e+01, -1.0000e+01,  2.0000e+01],
        [-1.4000e-04, -2.0000e+01,  2.0000e+01],
        [-1.4000e-04,  2.0000e+01,  2.0000e+01],
        [ 1.4000e-04, -2.0000e+01,  2.0000e+01],
        [ 1.4000e-04,  2.0000e+01,  2.0000e+01],
        [ 1.0000e+01,  1.0000e+01,  2.0000e+01],
        [ 2.0000e+01, -2.0000e+01, -2.0000e+01],
        [ 2.0000e+01, -2.0000e+01,  2.0000e+01],
        [ 2.0000e+01, -1.4000e-04,  2.0000e+01],
        [ 2.0000e+01,  1.4000e-04,  2.0000e+01],
        [ 2.0000e+01,  2.0000e+01, -2.0000e+01],
        [ 2.0000e+01,  2.0000e+01,  2.0000e+01],
        [-2.0000e+01, -1.4000e-04,  2.0000e+01],
        [-2.0000e+01,  0.0000e+00,  2.0000e+01],
        [

r_cfgs, a_cfgs, n_cfgs dicts interpolation & extrapolation

In [ ]:
def create_pure_torch_interpolated_cfg(data_dict, device="cuda"):
    """
    纯 PyTorch 架构版：100%在GPU端完成，支持任意连续频率张量的内插与两端线性外插
    """
    # 1. 将离散字典转化为有序的 GPU Tensor
    sorted_nus = sorted(data_dict.keys())
    sorted_vals = [data_dict[nu] for nu in sorted_nus]

    xp = torch.tensor(sorted_nus, dtype=torch.float32, device=device)
    fp = torch.tensor(sorted_vals, dtype=torch.float32, device=device)

    if len(xp) < 2:
        raise ValueError("至少需要2个已知点。")

    def torch_interp_extrap(nu_tensor):
        # 确保输入在正确的设备上
        nu = nu_tensor.to(device)

        # 2. 找到每个输入频率在已知序列中的桶索引（右边界）
        # indices 范围在 [0, len(xp)] 之间
        indices = torch.bucketize(nu, xp)

        # 3. 边界修正：将外插点的索引限制在合法的一级间距内
        # 如果超出右边界 (indices == len(xp))，使其借用最后一节的斜率进行右外插
        # 如果低于左边界 (indices == 0)，使其借用第一节的斜率进行左外插
        idx_right = torch.clamp(indices, 1, len(xp) - 1)
        idx_left = idx_right - 1

        # 4. 批量提取两端点的物理量
        x0, x1 = xp[idx_left], xp[idx_right]
        y0, y1 = fp[idx_left], fp[idx_right]

        # 5. 统一计算斜率并完成线性映射（数学上内插和线性外插共享此公式）
        slopes = (y1 - y0) / (x1 - x0)
        res = y0 + slopes * (nu - x0)

        return res

    return torch_interp_extrap

In [ ]:
# run test
import torch

# ---------------------------------------------------------
# Step 1: 准备你的离散实验测得的数据字典（例如某种特定玻璃或涂层）
# ---------------------------------------------------------
# 凸透镜的色散折射率数据 (频率: 折射率)
prism_raw_n = {
    4.0e14: 1.513,  # 红光附近
    5.5e14: 1.518,  # 绿光附近
    7.0e14: 1.524   # 蓝光附近
}

# 滤光片的特定光谱吸收率数据 (频率: 吸收系数)
filter_raw_a = {
    4.5e14: 0.02,  # 允许通行的波段：低吸收
    6.5e14: 0.85   # 需要截止的波段：强吸收
}

# ---------------------------------------------------------
# Step 2: 调用函数，生成高并发的 GPU 插值/外插算子
# ---------------------------------------------------------
# 此时，这两个变量已经变成了可以直接接收 PyTorch 连续张量的“连续函数”
prism_n_operator = create_pure_torch_interpolated_cfg(prism_raw_n, device=device)
filter_a_operator = create_pure_torch_interpolated_cfg(filter_raw_a, device=device)

# ---------------------------------------------------------
# Step 3: 将算子绑定到系统的物理材质配置表（n_cfgs, a_cfgs）
# ---------------------------------------------------------
# 假设通过你的 STL 载入函数，得知：
# 物体 ID [1] 是凸透镜 (collimator_lens) base
# 物体 ID [2] 是滤光片 (optical_filter) layer

n_configs = {
    1: 5.5, # prism_n_operator,                # 🟢 物体 1 的折射率使用我们的连续插值外插算子
    2: 2                            # 物体 222 依然可以使用普通的固定的常数标量
}

a_cfgs = {
    1: 0.01,                             # 物体 1 完美透明
    2: 0.01 # filter_a_operator                # 🟢 物体 2 的吸收率使用连续插值外插算子
}

r_cfgs = {
    1: 0.0,                            # 全场统一使用 4% 基础反射率常数
    2: 0.0
}

# ---------------------------------------------------------
# Step 4: 正常初始化并运行你的高级光线追踪引擎
# ---------------------------------------------------------
#实例化你的光线追踪器（内部带有 _evaluate_property 查表器）

ray_tracer = AdvancedRayTracerPBRTGPU(
    x_lim=[-500, 500], y_lim=[-500, 500], z_lim=[-500, 500],
    scene_intersector=intersector,  # 传入你之前构建好的 STL 求交器
    n_configs=n_configs,
    a_configs=a_cfgs,
    r_configs=r_cfgs,
    fresnel = True
)

# 发射全谱段光线进行追迹
# 这里的 nu_np 包含了从 3.0e14 到 8.0e14 的大量连续光线频率

#ray_orig = np.array([[ -10.,  -50., -300.]])
#ray_dir = np.array([[0., 0.18, 1.]])
#ray_frq = np.array([[550.]])

history, status = ray_tracer.trace(ray_orig, ray_dir, ray_frq, ray_pol)

d_hit_h: tensor([[0.8944, 0.0000, 0.4472]], device='cuda:0')
normals_h: tensor([[ 0.,  0., -1.]], device='cuda:0')
obj_ids_h: tensor([[0, 1]], device='cuda:0')
curr_medium_ids_h: tensor([0], device='cuda:0')
nu_h: tensor([[5.0000e+14]], device='cuda:0')
pol_h: tensor([[ 0., -1.,  0.]], device='cuda:0')
p_error_h: tensor([[1.0255e-05, 6.9605e-06, 1.2921e-05]], device='cuda:0')
directions_h: tensor([[0.8944, 0.0000, 0.4472]], device='cuda:0')
total_internal_reflection: tensor([False], device='cuda:0')
refraction
new_d_h: tensor([[0.1626, 0.0000, 0.9867]], device='cuda:0')
new_med_h: tensor([1], device='cuda:0')
reflect_mask_h: tensor([False], device='cuda:0')
transmit_mask_h: tensor([True], device='cuda:0')
absorb_mask_h: tensor([False], device='cuda:0')
bias_vector_h: tensor([[1.0255e-05, 6.9605e-06, 1.2921e-05]], device='cuda:0')
new_pol_h: tensor([[ 0., -1., -0.]], device='cuda:0')
d_hit_h: tensor([[0.1626, 0.0000, 0.9867]], device='cuda:0')
normals_h: tensor([[1., 0., 0.]], device='c

In [ ]:
# 在调用 trace 之前，打印这 9 条光线的初始方向
print("初始方向矩阵前几条:\n", ray_dir[:3])
# 检查模长是否为 1
print("初始方向模长:\n", torch.norm(torch.tensor(ray_dir), dim=-1))

初始方向矩阵前几条:
 [[0.8944276 0.        0.4472128]]
初始方向模长:
 tensor([1.])


In [ ]:
for i in history: print(i)

{'origins': tensor([[ -4.4721, -10.0000, -30.0000]], device='cuda:0'), 'directions': tensor([[0.8944, 0.0000, 0.4472]], device='cuda:0'), 'medium_ids': tensor([0], device='cuda:0'), 'polarization': tensor([[ 0., -1.,  0.]], device='cuda:0')}
{'origins': tensor([[ 15.5279, -10.0000, -20.0000]], device='cuda:0'), 'directions': tensor([[0.1626, 0.0000, 0.9867]], device='cuda:0'), 'medium_ids': tensor([1], device='cuda:0'), 'polarization': tensor([[ 0., -1., -0.]], device='cuda:0')}
{'origins': tensor([[ 20.0000, -10.0000,   7.1335]], device='cuda:0'), 'directions': tensor([[-0.1626,  0.0000,  0.9867]], device='cuda:0'), 'medium_ids': tensor([1], device='cuda:0'), 'polarization': tensor([[ 0., -1.,  0.]], device='cuda:0')}
{'origins': tensor([[ 17.8794, -10.0000,  20.0000]], device='cuda:0'), 'directions': tensor([[-0.8944,  0.0000,  0.4472]], device='cuda:0'), 'medium_ids': tensor([0], device='cuda:0'), 'polarization': tensor([[ 0., -1.,  0.]], device='cuda:0')}
{'origins': tensor([[  2.1

给我写个函数 输入为trace输出 输出为stl文件可以放到freecad中显示光线

In [ ]:
from stl import mesh

def export_ray_history_to_stl(ray_history, output_path="ray_tracks.stl", radius=0.05, num_sides=6):
    """
    🚀 极限全通道光线强转舱：无视任何物理状态，强行保障全场所有并发光线（如全部 9 条）完整生成 STL。
    """
    print("🎬 开始解析全量光线轨迹拓扑...")
    num_steps = len(ray_history)
    if num_steps < 2:
        print("❌ 历史记录步数不足，无法生成轨迹。")
        return

    # 1. 提取全量空间位置点大表
    history_origins = [step['origins'].detach().cpu().numpy() for step in ray_history]
    history_origins = np.stack(history_origins, axis=0)      # 形状: (Steps, Rays, 3)

    num_rays = history_origins.shape[1]
    print(f"📡 检测到矩阵内并行的逻辑光线总数: {num_rays} 条，正在全面释放几何膨胀...")

    all_cylinders_facets = []

    # 2. 构建圆柱体的本地拓扑模板
    angles = np.linspace(0, 2 * np.pi, num_sides, endpoint=False)
    cos_a = np.cos(angles)
    sin_a = np.sin(angles)

    # 3. 纵横全域遍历（彻底消除任何早夭 break）
    for r in range(num_rays):
        ray_has_drawn_segments = False  # 追踪当前光线是否成功绘制了至少一段网格

        for s in range(num_steps - 1):
            p0 = history_origins[s, r]
            p1 = history_origins[s + 1, r]

            v_ray = p1 - p0
            length = np.linalg.norm(v_ray)

            # 🟢 修正点 1：采用极致微小的机器精度级阈值 (1e-7)，防止微小位移光线被误杀。
            # 只有当两点在双精度下完全重合（即完全驻留在边界静止）时，才不画这一段。
            if length < 1e-7:
                continue

            # --- 罗德里格空间姿态对齐 ---
            z_axis = np.array([0.0, 0.0, 1.0])
            v_ray_norm = v_ray / length

            axis = np.cross(z_axis, v_ray_norm)
            axis_len = np.linalg.norm(axis)

            if axis_len < 1e-8:
                R = np.eye(3) if v_ray_norm[2] > 0 else -np.eye(3)
            else:
                axis = axis / axis_len
                angle = np.arccos(np.clip(np.dot(z_axis, v_ray_norm), -1.0, 1.0))
                K = np.array([[0, -axis[2], axis[1]],
                              [axis[2], 0, -axis[0]],
                              [-axis[1], axis[0], 0]])
                R = np.eye(3) + np.sin(angle) * K + (1 - np.cos(angle)) * np.dot(K, K)

            # 生成当前段圆柱体的底面和顶面顶点
            bottom_verts = np.zeros((num_sides, 3))
            top_verts = np.zeros((num_sides, 3))
            for i in range(num_sides):
                local_pt_bot = np.array([radius * cos_a[i], radius * sin_a[i], 0.0])
                local_pt_top = np.array([radius * cos_a[i], radius * sin_a[i], length])
                bottom_verts[i] = p0 + np.dot(R, local_pt_bot)
                top_verts[i] = p0 + np.dot(R, local_pt_top)

            # 缝合侧面三角面片
            for i in range(num_sides):
                next_i = (i + 1) % num_sides
                f1 = [bottom_verts[i], top_verts[i], top_verts[next_i]]
                f2 = [bottom_verts[i], top_verts[next_i], bottom_verts[next_i]]
                all_cylinders_facets.append(f1)
                all_cylinders_facets.append(f2)

            ray_has_drawn_segments = True

        # 🟢 修正点 2：极端防御保险（如果某条光线由于某种魔幻原因真的全程静止，长度全是0）
        # 我们强行给它在原位画一个微型的十字标记或者微小立方体，确保在 CAD 中绝对能数出 9 个实体！
        if not ray_has_drawn_segments:
            p_static = history_origins[0, r] # 拿到它出生点的坐标
            # 强行塞入一个小三角面片作为基准锚点，确保网格大表里有它的索引存在
            f_dummy = [p_static, p_static + np.array([radius, 0, 0]), p_static + np.array([0, radius, 0])]
            all_cylinders_facets.append(f_dummy)

    if len(all_cylinders_facets) == 0:
        print("⚠️ 未构建出任何有效面片。")
        return

    # 4. 打包封装并存储
    facets_np = np.array(all_cylinders_facets)
    ray_mesh = mesh.Mesh(np.zeros(facets_np.shape[0], dtype=mesh.Mesh.dtype))
    for i, facet in enumerate(facets_np):
        ray_mesh.v0[i] = facet[0]
        ray_mesh.v1[i] = facet[1]
        ray_mesh.v2[i] = facet[2]

    ray_mesh.save(output_path)
    print(f"🎉 全量转换成功！已强制将全场 {num_rays} 条光线轨迹导出至: {output_path} (共包含 {facets_np.shape[0]} 个三角面片)")

In [ ]:
export_ray_history_to_stl(history)

🎬 开始解析全量光线轨迹拓扑...
📡 检测到矩阵内并行的逻辑光线总数: 1 条，正在全面释放几何膨胀...
🎉 全量转换成功！已强制将全场 1 条光线轨迹导出至: ray_tracks.stl (共包含 108 个三角面片)


In [ ]:
import numpy as np
import torch

def save_trace_output_to_npz(ray_history, ray_status, file_path="ray_trace_data.npz"):
    """
    将打碎的 history 转换为连续的 N 维 NumPy 密集矩阵，并以 `.npz` 二进制压缩包形式存储
    """
    # 1. 瞬间将每一跳的 PyTorch 张量剥离 GPU 并转换为 NumPy 数组
    # 转换为物理形状: (Bounce_Steps, N_rays, 3)
    origins_all = np.stack([step['origins'].detach().cpu().numpy() for step in ray_history], axis=0)
    directions_all = np.stack([step['directions'].detach().cpu().numpy() for step in ray_history], axis=0)
    medium_ids_all = np.stack([step['medium_ids'].detach().cpu().numpy() for step in ray_history], axis=0)

    status_np = ray_status.detach().cpu().numpy()

    # 处理可能不开启的偏振态
    save_dict = {
        'origins': origins_all,
        'directions': directions_all,
        'medium_ids': medium_ids_all,
        'status': status_np
    }

    if ray_history[0]['polarization'] is not None:
        pols_all = np.stack([step['polarization'].detach().cpu().numpy() for step in ray_history], axis=0)
        save_dict['polarization'] = pols_all

    # 2. 🟢 核心：执行二进制无损压缩存储
    np.savez_compressed(file_path, **save_dict)
    print(f"🎉 跨平台二进制压缩包已生成: {file_path}")

# -------------------------------------------------------------
# 对应的读取函数（直接返还给你可以拿去画图或导入 FreeCAD 插件的数据）
def load_trace_output_from_npz(file_path="ray_trace_data.npz"):
    """
    读取 .npz 二进制文件
    """
    with np.load(file_path) as data:
        origins = data['origins']        # 形状: (Steps, Rays, 3)
        directions = data['directions']  # 形状: (Steps, Rays, 3)
        medium_ids = data['medium_ids']  # 形状: (Steps, Rays)
        status = data['status']          # 形状: (Rays,)
        polarization = data['polarization'] if 'polarization' in data.files else None

    return origins, directions, medium_ids, polarization, status

In [ ]:
print(history[0]['origins'].shape)


torch.Size([9, 3])


In [ ]:
# 执行你的高并发光学仿真
history, status = ray_tracer.trace(origins_np, directions_np, nu_np, polarization_np)

# 💾 一键保存到硬盘
save_trace_output_binary((history, status), "lens_simulation_001.pt")

# 📂 到了第二天，或者在另一个脚本里一键恢复
# 甚至可以直接指定加载到 CPU 上以便节省宝贵的显存
history_loaded, status_loaded = load_trace_output_binary("lens_simulation_001.pt", device="cpu")

给我写个freecad插件 输入为trace输出的二进制文件 在freecad中显示光线 然后双击选中了每条光线，显示这个光线的偏振（如果有），频率，方向，还有所在介质名称和折射率；然后右键可以选择显示它所属的这一条光线的全部轨迹并隐藏其他所有光线

结合上一问我们编写的两种二进制保存方案，最适合在 FreeCAD 中直接作为数据源读取的是方案二（.npz 格式）。因为 FreeCAD 内部自带的 Python 环境非常干净（基于标准 Python + PySide），默认没有安装、也不需要安装重型的 PyTorch 库。

通过 .npz 二进制压缩包，FreeCAD 只需要依靠自带的 numpy 就能以极快的速度恢复出全量高维矩阵，进而生成 3D 交互光轨。

下面是为您量身定制的 FreeCAD 插件宏代码（完整集成版）。它将二进制读取、3D 拓扑构建、高维物理属性注入完美融合在一起。

1. FreeCAD 交互式二进制光轨加载插件代码
请在 FreeCAD 中点击 宏 (Macros) -> 宏... -> 新建，命名为 BinaryRayImporter.py，然后将以下代码完整粘贴进去并保存：

In [ ]:
import FreeCAD as App
import FreeCADGui as Gui
import Part
import numpy as np
import os

def import_ray_npz_to_freecad(npz_path, id_to_name=None, n_configs=None):
    """
    从 .npz 二进制文件中读取光线追踪结果，并在 FreeCAD 中构建交互式高维物理模型

    :param npz_path: str, .npz 二进制文件的绝对路径
    :param id_to_name: dict, 可选，ID 到物体名称的映射，如 {1: 'Lens_A'}
    :param n_configs: dict, 可选，材质折射率配置，如果提供常数，可直接反查
    """
    if not os.path.exists(npz_path):
        App.Console.PrintError(f"❌ 找不到指定的二进制文件: {npz_path}\n")
        return

    App.Console.PrintMessage(f"💾 正在从二进制文件加载光轨流: {npz_path} ...\n")

    # 1. 解包二进制密集矩阵数据
    with np.load(npz_path) as data:
        origins = data['origins']        # (Steps, Rays, 3)
        directions = data['directions']  # (Steps, Rays, 3)
        medium_ids = data['medium_ids']  # (Steps, Rays)
        status = data['status']          # (Rays,)
        # 频率和偏振是可选通道
        frequencies = data['nu'] if 'nu' in data.files else None # (Steps, Rays) 或 (Rays,)
        polarizations = data['polarization'] if 'polarization' in data.files else None # (Steps, Rays, 3)

    num_steps, num_rays, _ = origins.shape

    doc = App.activeDocument()
    if doc is None:
        doc = App.newDocument("Optical_Scene_From_Binary")

    # 2. 创建主图层组
    main_group = doc.addObject("App::DocumentObjectGroup", "Ray_Scene_Tracks")
    main_group.Label = f"归档光轨 ({os.path.basename(npz_path)})"

    App.Console.PrintMessage(f"🚀 正在并行构建 {num_rays} 条光子的空间物理流...\n")

    # 3. 展开高维矩阵，进行实体缝合与属性硬编码
    for r in range(num_rays):
        # 为单条光线建立专属轨迹线段容器
        ray_group = doc.addObject("App::DocumentObjectGroup", f"Ray_{r:04d}")
        ray_group.Label = f"光线 【Ray {r}】"
        main_group.addObject(ray_group)

        for s in range(num_steps - 1):
            p0 = origins[s, r]
            p1 = origins[s+1, r]
            d0 = directions[s, r]
            m0 = int(medium_ids[s, r])

            # 物理边界过滤：如果方向向量全零，说明在上一跳已被吸收或寿终正寝
            if np.allclose(d0, 0.0, atol=1e-6):
                break

            # 几何边界过滤：如果没有移动，不生成冗余线段
            if np.allclose(p0, p1, atol=1e-5):
                continue

            # 4. 在 FreeCAD 3D 内核中绘制线段
            v0 = App.Vector(p0[0], p0[1], p0[2])
            v1 = App.Vector(p1[0], p1[1], p1[2])
            line_seg = Part.LineSegment(v0, v1)

            seg_obj = doc.addObject("Part::Feature", f"R{r}_Step{s}")
            seg_obj.Shape = line_seg.toShape()
            ray_group.addObject(seg_obj)

            # 5. 🚀 关键：利用 FreeCAD 原生动态属性机制，将高维物理元数据原地封印到节点中 🚀
            seg_obj.addProperty("App::PropertyString", "RayID", "光子核心身份").RayID = f"物理光子 #{r}"
            seg_obj.addProperty("App::PropertyString", "BounceStep", "光子核心身份").BounceStep = f"第 {s} 次碰撞弹跳"

            # 写入 3D 方向矢量
            seg_obj.addProperty("App::PropertyVector", "Direction", "运动学特性").Direction = App.Vector(d0[0], d0[1], d0[2])

            # 写入宿主介质名称与 ID
            med_name = id_to_name.get(m0, "外场背景(真空/空气)") if id_to_name else f"未知介质 ID: {m0}"
            seg_obj.addProperty("App::PropertyString", "MediumName", "宿主空间介质").MediumName = med_name

            # 写入折射率数值（优先反查配置，否则给默认）
            n_val = 1.0
            if n_configs and m0 in n_configs:
                if not callable(n_configs[m0]): # 标量常数直读
                    n_val = float(n_configs[m0])
            seg_obj.addProperty("App::PropertyFloat", "RefractiveIndex", "宿主空间介质").RefractiveIndex = n_val

            # 写入物理频率
            if frequencies is not None:
                nu_val = float(frequencies[s, r]) if frequencies.ndim == 2 else float(frequencies[r])
                seg_obj.addProperty("App::PropertyFloat", "Frequency_Hz", "波谱物理属性").Frequency_Hz = nu_val

            # 写入偏振电场矢量（如果有）
            if polarizations is not None:
                pol0 = polarizations[s, r]
                seg_obj.addProperty("App::PropertyVector", "PolarizationVector", "麦克斯韦偏振态").PolarizationVector = App.Vector(pol0[0], pol0[1], pol0[2])
                pol_amp = float(np.linalg.norm(pol0))
                seg_obj.addProperty("App::PropertyFloat", "FieldAmplitude", "麦克斯韦偏振态").FieldAmplitude = pol_amp

            # 6. 渲染美化：设置高可见度红色线宽
            if hasattr(seg_obj, "ViewObject"):
                seg_obj.ViewObject.LineColor = (1.0, 0.1, 0.1)
                seg_obj.ViewObject.LineWidth = 2.0

    doc.recompute()
    App.Console.PrintMessage("🎉 二进制光轨加载成功！高维物理属性已成功注入属性树面板。\n")

# --- 快捷测试执行入口 ---
# 替换为你实际导出的 .npz 文件路径
# 映射表可以直接写死，用来把数字 ID 翻译成优雅的透镜人类文字
id_mapping = {1: "凸透镜 (BK7_Lens)", 2: "凹透镜 (Flint_Lens)", 3: "样品比色皿 (Quartz)"}
n_table = {1: 1.5168, 2: 1.6200, 3: 1.458}

# 在 FreeCAD 中执行加载
# import_ray_npz_to_freecad("D:/optics_output/ray_trace_data.npz", id_to_name=id_mapping, n_configs=n_table)

2. 双击查看高维物理状态运行此宏后，FreeCAD 会根据矩阵流自动绘制出细致的 3D 线段。用户双击 3D 视图中的任意一段光轨，或者直接在左侧模型树点击单段线段，FreeCAD 的左下角属性编辑器面板（Property Editor）会完美展现如下分层表格：分组名称属性键实时查验的数值示例光子核心身份RayID物理光子 #142BounceStep第 2 次碰撞弹跳运动学特性Direction(0.12, 0.54, 0.83)宿主空间介质MediumName凸透镜 (BK7_Lens)RefractiveIndex1.5168波谱物理属性Frequency_Hz5.45e+14麦克斯韦偏振态PolarizationVector(0.707, -0.707, 0.00)FieldAmplitude1.00003. 右键交互隔离全轨迹（一键过滤宏）为了实现您要求的“一键只看当前光线全轨迹，隐藏全场其余所有光线”的右键过滤功能，我们在 FreeCAD 中使用快捷键或独立动作宏来实现最佳交互。请在 FreeCAD 宏管理器中新建第二个宏，命名为 IsolateRayTrack.py：

In [ ]:
import FreeCAD as App
import FreeCADGui as Gui

def isolate_current_ray_workflow():
    """
    一键孤立当前选中的光线，隐藏全场其他所有噪声光线
    """
    # 1. 获取当前用户选中的线段对象
    selected_objs = Gui.Selection.getSelection()
    if not selected_objs:
        App.Console.PrintMessage("💡 请先在 3D 视图或模型树中选中某条光线的任意一段！\n")
        return

    target_seg = selected_objs[0]

    # 2. 向上追溯，寻找其属于哪一条光线组 (例如 Ray_0142)
    belonging_ray_group = None
    for parent in target_seg.InList:
        if parent.Name.startswith("Ray_"):
            belonging_ray_group = parent
            break

    if belonging_ray_group is None:
        App.Console.PrintMessage("❌ 选中的几何体不是由该光学引擎生成的光线段。\n")
        return

    doc = App.activeDocument()
    main_container = doc.getObject("Ray_Scene_Tracks")
    if not main_container:
        return

    # 3. 矩阵式一键切换全局 Visibility（可见性状态）
    App.Console.PrintMessage(f"✂️ 正在隔离展现 {belonging_ray_group.Label} 的全寿命周期轨迹...\n")

    # 挂起界面刷新，提升批量隐藏的速度
    Gui.Control.lazyRecompute(False)

    for ray_node in main_container.OutList:
        if ray_node.Name == belonging_ray_group.Name:
            # 目标全轨迹整体点亮
            ray_node.ViewObject.show()
            for sub_seg in ray_node.OutList:
                sub_seg.ViewObject.show()
        else:
            # 其余非关联光线全族一键隐藏
            ray_node.ViewObject.hide()

    Gui.Control.lazyRecompute(True)
    doc.recompute()

# 执行孤立隔离
isolate_current_ray_workflow()

🛠️ 最佳交互配置方法：
由于 FreeCAD 原生右键菜单的修改需要改动系统的底层 Qt 底座，在实际科研和工程流中，将其绑定为键盘单键快捷键（例如键盘上的 I 键，代表 Isolate）是最顺手且高效的做法：

在 FreeCAD 顶部菜单点击 工具 (Tools) -> 自定义 (Customize)。

切换到 键盘 (Keyboard) 标签页。

在左侧分类中选择 Macros（宏），找到你刚才保存的 IsolateRayTrack。

在右侧的 快捷键 (Shortcut) 输入框里按下键盘上的 I 键（或者您喜欢的任意键），点击 分配 (Assign)。

此后的交互爽快感将大幅提升：在 FreeCAD 庞大的光学系统 3D 模型中，你只需转动鼠标，双击某一段感兴趣的光线。左下角会立刻显示出它当时精确的偏振态矢量和所在透镜的折射率。此时轻敲一下键盘上的 I 键，轰！全场几万条乱飞的光线瞬间蒸发消失，只留下这一条特定光子从诞生、穿过各个 STL 透镜发生折射弯曲、到最终命运的清晰单线条完整轨迹！

* gpu contains method

In [ ]:
!pip install trimesh

  Using cached trimesh-4.12.2-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 741.0/741.0 kB 16.8 MB/s eta 0:00:00


In [ ]:
import torch

def ray_triangle_intersect(orig, dir, v0, v1, v2, eps=1e-8):
    # orig: (N, 3)
    # dir:  (N, 3)
    # v0,v1,v2: (F, 3)

    e1 = v1 - v0
    e2 = v2 - v0

    pvec = torch.cross(dir[:, None, :], e2[None, :, :], dim=-1)
    det = torch.sum(e1[None, :, :] * pvec, dim=-1)

    mask = det.abs() > eps
    inv_det = torch.zeros_like(det)
    inv_det[mask] = 1.0 / det[mask]

    tvec = orig[:, None, :] - v0[None, :, :]
    u = torch.sum(tvec * pvec, dim=-1) * inv_det
    mask_u = (u >= 0) & (u <= 1)

    qvec = torch.cross(tvec, e1[None, :, :], dim=-1)
    v = torch.sum(dir[:, None, :] * qvec, dim=-1) * inv_det
    mask_v = (v >= 0) & (u + v <= 1)

    t = torch.sum(e2[None, :, :] * qvec, dim=-1) * inv_det
    mask_t = t > eps

    hit = mask & mask_u & mask_v & mask_t
    return hit

def point_in_mesh(points, vertices, faces, device="cuda"):
    points = torch.tensor(points, dtype=torch.float32, device=device)
    vertices = torch.tensor(vertices, dtype=torch.float32, device=device)
    faces = torch.tensor(faces, dtype=torch.long, device=device)

    v0 = vertices[faces[:, 0]]
    v1 = vertices[faces[:, 1]]
    v2 = vertices[faces[:, 2]]

    N = points.shape[0]
    votes = torch.zeros(N, dtype=torch.int32, device=device)

    base_dirs = torch.tensor([
        [1,0,0],
        [0,1,0],
        [0,0,1],
    ], dtype=torch.float32, device=device)

    for d in base_dirs:
        eps = (torch.rand(3, device=device) - 0.5) * 1e-4
        dir_vec = (d + eps)
        dir_vec = dir_vec / torch.norm(dir_vec)

        dirs = dir_vec[None, :].expand(N, 3)

        hit = ray_triangle_intersect(points, dirs, v0, v1, v2)
        hit_count = hit.sum(dim=1)

        votes += (hit_count % 2 == 1).int()

    return (votes >= 2)


In [ ]:
import numpy as np

# --- 立方体 8 个顶点 ---
verts_np = np.array([
    [-1, -1, -1],  # 0
    [ 1, -1, -1],  # 1
    [ 1,  1, -1],  # 2
    [-1,  1, -1],  # 3
    [-1, -1,  1],  # 4
    [ 1, -1,  1],  # 5
    [ 1,  1,  1],  # 6
    [-1,  1,  1],  # 7
], dtype=np.float32)

# --- 立方体 12 个三角面（每个面 2 个三角形） ---
faces_np = np.array([
    [0, 1, 2], [0, 2, 3],   # bottom (z = -1)
    [4, 5, 6], [4, 6, 7],   # top    (z = +1)
    [0, 1, 5], [0, 5, 4],   # front  (y = -1)
    [2, 3, 7], [2, 7, 6],   # back   (y = +1)
    [1, 2, 6], [1, 6, 5],   # right  (x = +1)
    [0, 3, 7], [0, 7, 4],   # left   (x = -1)
], dtype=np.int32)

# --- 生成三个点 ---
point_inside  = np.array([0.0, 0.0, 0.0], dtype=np.float32)   # 在 cube 内
point_on      = np.array([1.0, 0.0, 0.0], dtype=np.float32)   # 在 cube 表面
point_outside = np.array([2.0, 0.0, 0.0], dtype=np.float32)   # 在 cube 外

points_np = np.stack([point_inside, point_on, point_outside], axis=0)


In [ ]:
inside = point_in_mesh(points_np, verts_np, faces_np)
print(inside)


tensor([ True, False, False], device='cuda:0')
